# ConformalOrb — the complete pipeline in one notebook (Kaggle)

From nothing to every number in the paper: Phase 0 verification → QH9-stable acquisition → official-split subset → survey →
QHNet inference → one-Fock-build test-time features → conformal certificates and FDR-controlled selection → SCF cost and the
chemistry test. Every stage writes its artifacts under `/kaggle/working/conformalorb/` and **skips itself when its artifacts already
exist** — either in the working directory or in an attached input (a previous version of this very notebook, or the outputs of the
per-phase notebooks). Heavy stages are also resumable row by row. So the whole thing runs across several 12-hour sessions:
run → *Save Version → Quick Save (Save output)* → open the notebook again → *Add Input → Your Work → previous version* → run.

| Stage | What | Resource / time (Kaggle) | Artifacts |
|---|---|---|---|
| 0 | Phase 0 verification on 8 molecules (gauge invariance of $s_B$, Weyl / Davis–Kahan) | CPU, 3 min | `phase0/` |
| 1 | QH9-stable database: attached → local → **Zenodo** (parallel resumable parts or single-stream) | CPU, ~30–120 min, 60 GB scratch in `/kaggle/tmp` | (scratch) |
| 2 | Compact subset of the official held-out splits (id-test 13,084 ∪ ood-test 9,335, upper-triangular Fock, lossless) | CPU, 5 min | `ConformalOrb_subset.db` |
| 3 | Survey of 20,000 molecules: conditioning of $S$, degeneracy, Davis–Kahan admissibility, Mondrian bins | CPU, ~25 min | `qh9_survey.csv/.png` |
| 4 | QHNet inference (official checkpoints) → $\hat H$, frontier errors, MO-basis error localisation, PT estimates | **GPU T4** 95 min (CPU: ~6 h) | `ConformalOrb_pred.db`, `phase1b_errors.csv` |
| 5 | One Fock build per prediction: self-consistency residual, $F_1$-vs-$\hat H$ frontier comparison, PT-informed error estimates | CPU, 4 workers, density fitting: ~4 h for 10k id + 6k ood | `phase2_features.csv` |
| 6 | Conformal certificates ($\hat H$ and one-step-corrected $F_1$), Mondrian / PT-normalised variants, conformal selection with FDR control, routing | CPU, minutes | `phase2_summary.csv`, figures |
| 7 | SCF cost (default vs warm starts) on a sample; frontier-orbital Fukui / site ranking for all predictions | CPU, ~3–4 h | `phase3_*.csv`, figure |

**Settings:** Internet on. Accelerator: GPU only for the session that runs stage 4 (all other stages are CPU; a GPU session is fine
for everything if quota allows). Level of theory everywhere: B3LYP-VWN5 (`b3lyp5`) / def2-SVP — the exact QH9 setting.

**Smoke test:** set `DATA_SOURCE = "synthetic"` — an 8-molecule database is computed in place of QH9 and every stage runs in ~10 minutes.


## Parameters

In [1]:
import os, glob
IN_KAGGLE = os.path.exists("/kaggle")

# ------------------------------------------------------------------ which stages to run (a stage is skipped anyway when its artifacts exist)
STAGES = {"phase0": True, "acquire": True, "subset": True, "survey": True, "infer": True, "fock": True, "conformal": True, "phase3": True}
DATA_SOURCE   = "qh9"          # "qh9" = the real QH9-stable database | "synthetic" = 8-molecule smoke test, no download

# ------------------------------------------------------------------ stage 0
DK_TRIALS     = 20
# ------------------------------------------------------------------ stage 1 (acquisition)
ACQUIRE_MODE  = "auto"         # "parts" (parallel, resumable, ~57 GB peak) | "stream" (single connection, ~30 GB peak) | "auto"
N_CONN        = 8              # concurrent range connections per archive part in "parts" mode
VERIFY_MD5    = True
# ------------------------------------------------------------------ stage 2 (subset)
SUBSET_N_ID   = 0              # 0 = all 13,084 id-test molecules
SUBSET_N_OOD  = 0              # 0 = all 9,335 ood-test molecules
# ------------------------------------------------------------------ stage 3 (survey)
N_SAMPLE      = 20000
CHECK_SCF     = 3
# ------------------------------------------------------------------ stage 4 (inference)
INFER_MAX     = {"id": 0, "ood": 0}      # 0 = all molecules of the split
SELF_CHECK_N  = 3
# ------------------------------------------------------------------ stage 5 (one Fock build)
FOCK_MAX      = {"id": 10000, "ood": 6000}   # per checkpoint (0 = all); 16k rows took ~8 h of CPU sessions
FOCK_WORKERS  = 4
FOCK_USE_DF   = True           # density-fitted J/K (~2.5x faster; ||F_DF - F_exact|| ~ 1e-3 Ha = 1-5% of a clean residual)
FOCK_GRID     = 1
# ------------------------------------------------------------------ stage 6 (conformal)
ALPHAS        = (0.10, 0.05)
R_REPEATS     = 20
TAU           = {"eps_HOMO": 2e-3, "eps_LUMO": 5e-3, "gap": 5e-3, "sB_HOMO": 0.1, "sB_LUMO": 0.2}
TAU_GROSS     = {"eps_HOMO": 1e-2, "eps_LUMO": 2e-2, "gap": 2e-2, "sB_HOMO": 0.7, "sB_LUMO": 0.7}
FDR_LEVELS    = (0.002, 0.005, 0.01, 0.02, 0.05)
Q_ROUTE       = 0.005
# ------------------------------------------------------------------ stage 7 (cost + chemistry)
N_SCF_PER_CKPT = 100
FUKUI_ALL      = True
SEED          = 20260916

OUT = "/kaggle/working/conformalorb" if IN_KAGGLE else "conformalorb_out"
SCRATCH = "/kaggle/tmp" if IN_KAGGLE else "scratch"


## Setup — dependencies, output layout, resume from attached inputs

In [2]:
%pip install -q pyscf threadpoolctl e3nn torch_geometric gdown
import os, sys, csv, time, math, json, shutil, sqlite3, warnings, subprocess, urllib.request
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
for d in (OUT, SCRATCH, f"{OUT}/phase0", f"{OUT}/phase2", f"{OUT}/phase3", f"{OUT}/checkpoints", "models"):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.getcwd())

def find_input(name):
    """Look for an artifact in the working output first, then in any attached input (previous versions, per-phase notebooks)."""
    for pat in (f"{OUT}/**/{name}", f"/kaggle/input/**/{name}" if IN_KAGGLE else f"**/{name}"):
        hits = sorted(glob.glob(pat, recursive=True))
        if hits: return hits[0]
    return None

def resume(name, sub="", writable=False):
    """Return a path to the artifact: in OUT if present; else copy (writable) or use (read-only) an attached one; else None."""
    dst = os.path.join(OUT, sub, name) if sub else os.path.join(OUT, name)
    if os.path.exists(dst): return dst
    src = find_input(name)
    if src is None: return None
    if writable:
        shutil.copy(src, dst); print(f"resumed {name} from {src}"); return dst
    return src

def db_rows(path, table, where=""):
    try:
        c = sqlite3.connect(f"file:{path}?mode=ro", uri=True); n = c.execute(f"SELECT COUNT(*) FROM {table} {where}").fetchone()[0]; c.close(); return n
    except Exception: return 0

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(stage 4 will run on CPU: ~1 s/molecule)")
print("free scratch:", f"{shutil.disk_usage(SCRATCH).free/1e9:.0f} GB")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.7/450.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 56.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
device: cuda Tesla T4
free scratch: 1100 GB


## Modules (written to disk so that worker processes can import them)

In [3]:
%%writefile co_acquire.py
# ConformalOrb: QH9 acquisition (Zenodo parts / streaming), tested Sept 2026
import io, os, sys, glob, shutil, hashlib, zipfile, time, urllib.request, urllib.error

ZENODO_RECORD = "8274793"
ZENODO_PARTS = {           # name -> (md5, approx size GB)  from the Zenodo record page
    "QH9Stable.zip.001": ("fa7d85e4dd0ec31bf030fdf9b9b35f6b", 8.5),
    "QH9Stable.zip.002": ("707c6230e2453ca9d3f0e3b75aa08e26", 8.5),
    "QH9Stable.zip.003": ("49765940355db946e10ac9a742544a77", 8.5),
    "QH9Stable.zip.004": ("091651e7ab708f383803c9373cef44e0", 1.4),
}
ZENODO_URL = "https://zenodo.org/records/{rec}/files/{name}?download=1"


def _fmt(nbytes):
    return f"{nbytes/1e9:.2f} GB"


def download_resumable(url, path, chunk=1 << 22, max_retries=8):
    """HTTP download with byte-range resume. Safe to re-run; picks up where it stopped."""
    have = os.path.getsize(path) if os.path.exists(path) else 0
    for attempt in range(max_retries):
        req = urllib.request.Request(url, headers={"User-Agent": "conformalorb/1.0"})
        if have:
            req.add_header("Range", f"bytes={have}-")
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                status = resp.getcode()
                if have and status == 200:            # server ignored Range: start over
                    have = 0; mode = "wb"
                else:
                    mode = "ab" if have else "wb"
                total = resp.headers.get("Content-Length")
                total = have + int(total) if total else None
                t0, done_now = time.time(), 0
                with open(path, mode) as fh:
                    while True:
                        buf = resp.read(chunk)
                        if not buf:
                            break
                        fh.write(buf); have += len(buf); done_now += len(buf)
                        if total and (done_now // chunk) % 50 == 0:
                            rate = done_now / max(time.time() - t0, 1e-6)
                            print(f"\r    {os.path.basename(path)}: {_fmt(have)} / {_fmt(total)}"
                                  f"  ({rate/1e6:.0f} MB/s)", end="", flush=True)
                if done_now: print()
                if total is None or have >= total:
                    return path
        except urllib.error.HTTPError as e:
            if e.code == 416:                          # range not satisfiable: already complete
                return path
            print(f"\n    HTTP {e.code} on attempt {attempt+1}; retrying in 10 s"); time.sleep(10)
        except Exception as e:                         # dropped connection etc.
            print(f"\n    {type(e).__name__}: {e}; resuming in 10 s (attempt {attempt+1})"); time.sleep(10)
    raise RuntimeError(f"download failed after {max_retries} attempts: {url}")


def md5sum(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as fh:
        for buf in iter(lambda: fh.read(chunk), b""):
            h.update(buf)
    return h.hexdigest()


class MultiPartFile(io.RawIOBase):
    """Read-only, seekable view over the byte-concatenation of several files.
    Lets zipfile read a byte-split archive (x.zip.001, .002, ...) without joining it on disk."""
    def __init__(self, paths):
        self.paths = list(paths); self.sizes = [os.path.getsize(p) for p in self.paths]
        self.starts = [sum(self.sizes[:i]) for i in range(len(self.sizes))]
        self.total = sum(self.sizes); self.pos = 0; self._fh = None; self._idx = None
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos
    def seek(self, off, whence=io.SEEK_SET):
        if whence == io.SEEK_SET: self.pos = off
        elif whence == io.SEEK_CUR: self.pos += off
        elif whence == io.SEEK_END: self.pos = self.total + off
        self.pos = max(0, min(self.pos, self.total)); return self.pos
    def _open(self, idx):
        if self._idx != idx:
            if self._fh: self._fh.close()
            self._fh = open(self.paths[idx], "rb"); self._idx = idx
        return self._fh
    def readinto(self, b):
        n = len(b); out = 0
        while out < n and self.pos < self.total:
            idx = max(i for i, s in enumerate(self.starts) if s <= self.pos)
            fh = self._open(idx); fh.seek(self.pos - self.starts[idx])
            got = fh.readinto(memoryview(b)[out:out + min(n - out, self.starts[idx] + self.sizes[idx] - self.pos)])
            if not got: break
            out += got; self.pos += got
        return out
    def close(self):
        if self._fh: self._fh.close()
        super().close()


def extract_db_from_parts(part_paths, dest_dir):
    """Stream the .db member(s) out of the split zip. Returns the path of the .db file."""
    os.makedirs(dest_dir, exist_ok=True)
    found = None
    with zipfile.ZipFile(io.BufferedReader(MultiPartFile(part_paths), buffer_size=1 << 24)) as zf:
        members = [m for m in zf.infolist() if not m.is_dir()]
        print("    archive members:", [(m.filename, _fmt(m.file_size)) for m in members])
        for m in members:
            out = os.path.join(dest_dir, os.path.basename(m.filename))
            if os.path.exists(out) and os.path.getsize(out) == m.file_size:
                print(f"    {out} already extracted"); 
            else:
                t0 = time.time()
                with zf.open(m) as src, open(out, "wb") as dst:
                    shutil.copyfileobj(src, dst, length=1 << 24)
                print(f"    extracted {out} ({_fmt(m.file_size)}) in {time.time()-t0:.0f} s")
            if out.endswith(".db") and (found is None or os.path.basename(out) == "QH9Stable.db"):
                found = out
    return found


def db_looks_valid(path):
    """Cheap probe: SQLite header + the QH9 `data` table answers a primary-key query."""
    if not os.path.exists(path) or os.path.getsize(path) < 1 << 20:
        return False
    try:
        import sqlite3
        with open(path, "rb") as fh:
            if fh.read(16) != b"SQLite format 3\x00":
                return False
        con = sqlite3.connect(f"file:{path}?mode=ro", uri=True)
        n = con.execute("SELECT max(id) FROM data").fetchone()[0]; con.close()
        return n is not None and n > 0
    except Exception:
        return False


def get_qh9_database(data_dir, drive_dir=None, verify_md5=True, persist_to_drive=False,
                     delete_parts=True, min_free_gb=60, n_conn=8):
    """Return a local path to QH9Stable.db, obtaining it in this order:
       1. already in data_dir
       2. copy from a mounted Google Drive folder (drive_dir/QH9Stable.db)
       3. download the split archive from Zenodo, verify, extract
    Never raises on a download problem: prints what to do and returns None."""
    os.makedirs(data_dir, exist_ok=True)
    local = os.path.join(data_dir, "QH9Stable.db")
    if db_looks_valid(local):
        print(f"[1] using local {local} ({_fmt(os.path.getsize(local))})"); return local

    if drive_dir:
        cand = os.path.join(drive_dir, "QH9Stable.db")
        if os.path.exists(cand):
            print(f"[2] copying from Drive: {cand} ({_fmt(os.path.getsize(cand))}) ...")
            t0 = time.time(); shutil.copyfile(cand, local)
            print(f"    done in {time.time()-t0:.0f} s"); return local

    free = shutil.disk_usage(data_dir).free / 1e9
    print(f"[3] Zenodo download; free disk in {data_dir}: {free:.0f} GB (need ~{min_free_gb} GB peak)")
    if free < min_free_gb:
        print("    NOT ENOUGH DISK. Options: use a Colab runtime with more disk, or download the parts\n"
              "    to Google Drive from https://zenodo.org/records/8274793 and set DRIVE_DIR.")
        return None
    parts = []
    for name, (md5, gb) in ZENODO_PARTS.items():
        p = os.path.join(data_dir, name)
        print(f"    {name} (~{gb} GB)")
        try:
            download_parallel(ZENODO_URL.format(rec=ZENODO_RECORD, name=name), p, n_conn=n_conn, fallback=download_resumable)
        except Exception as e:
            print(f"    download failed: {e}\n    re-run this cell to resume, or fetch the parts manually from\n"
                  f"    https://zenodo.org/records/{ZENODO_RECORD} into {data_dir}")
            return None
        if verify_md5:
            got = md5sum(p)
            if got != md5:
                print(f"    md5 MISMATCH for {name}: {got} != {md5}. Deleting the part; re-run to download again.")
                os.remove(p); return None
            print(f"    md5 ok")
        parts.append(p)

    print("    extracting across the 4 parts (no re-assembly on disk) ...")
    db = extract_db_from_parts(parts, data_dir)
    if db is None:
        print("    no .db member found in the archive"); return None
    if db != local:
        os.replace(db, local)
    if delete_parts:
        for p in parts: os.remove(p)
        print("    parts deleted")
    if persist_to_drive and drive_dir:
        os.makedirs(drive_dir, exist_ok=True); dst = os.path.join(drive_dir, "QH9Stable.db")
        print(f"    copying to Drive for future sessions: {dst} ..."); shutil.copyfile(local, dst)
    print(f"    ready: {local} ({_fmt(os.path.getsize(local))})")
    return local


import json, os, threading, time, urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed


def _probe(url):
    """(total_size, supports_ranges) via a 1-byte range request."""
    req = urllib.request.Request(url, headers={"Range": "bytes=0-0", "User-Agent": "conformalorb/1.0"})
    with urllib.request.urlopen(req, timeout=60) as r:
        cr = r.headers.get("Content-Range")
        if r.getcode() == 206 and cr and "/" in cr:
            return int(cr.split("/")[-1]), True
        cl = r.headers.get("Content-Length")
        return (int(cl) if cl else None), False


def download_parallel(url, path, n_conn=8, segment=256 << 20, max_retries=12, fallback=None,
                      _test_fail_once=()):
    """Download `url` to `path` with `n_conn` concurrent range requests of `segment` bytes.
    Resumable: finished segments are recorded in path+'.progress.json'. Returns path."""
    try:
        total, ok = _probe(url)
    except Exception as e:
        total, ok = None, False
        print(f"    probe failed ({e}); falling back to a single connection")
    if not ok or not total:
        if fallback is None:
            raise RuntimeError("server does not support byte ranges and no fallback given")
        return fallback(url, path)

    prog = path + ".progress.json"
    done = set()
    if os.path.exists(path) and os.path.getsize(path) == total and os.path.exists(prog):
        done = set(json.load(open(prog)))
    else:
        with open(path, "wb") as fh:
            fh.truncate(total)                          # sparse pre-allocation
    n_seg = (total + segment - 1) // segment
    todo = [k for k in range(n_seg) if k not in done]
    print(f"    {os.path.basename(path)}: {total/1e9:.2f} GB in {n_seg} segments, "
          f"{len(done)} already done, {n_conn} connections")
    if not todo:
        os.remove(prog); return path

    fd = os.open(path, os.O_RDWR); lock = threading.Lock()
    got = {"bytes": 0}; failed_once = set(_test_fail_once); t0 = time.time()

    def fetch(k):
        a, b = k * segment, min(total, (k + 1) * segment) - 1
        for attempt in range(max_retries):
            try:
                if k in failed_once:                    # test hook: fail this segment exactly once
                    failed_once.discard(k); raise ConnectionResetError("simulated")
                req = urllib.request.Request(url, headers={"Range": f"bytes={a}-{b}", "User-Agent": "conformalorb/1.0"})
                with urllib.request.urlopen(req, timeout=120) as r:
                    if r.getcode() != 206:
                        raise RuntimeError("expected 206")
                    off = a
                    while True:
                        buf = r.read(4 << 20)
                        if not buf:
                            break
                        os.pwrite(fd, buf, off); off += len(buf)
                        with lock:
                            got["bytes"] += len(buf)
                    if off != b + 1:
                        raise RuntimeError(f"short segment {k}: {off - a} of {b - a + 1} bytes")
                with lock:
                    done.add(k); json.dump(sorted(done), open(prog, "w"))
                return k
            except urllib.error.HTTPError as e:
                wait = 30 if e.code == 429 else min(60, 5 * (attempt + 1))
                time.sleep(wait)
            except Exception:
                time.sleep(min(60, 5 * (attempt + 1)))
        raise RuntimeError(f"segment {k} failed after {max_retries} attempts")

    with ThreadPoolExecutor(max_workers=n_conn) as ex:
        futs = {ex.submit(fetch, k): k for k in todo}
        last = time.time()
        for f in as_completed(futs):
            f.result()                                  # re-raise any failure
            if time.time() - last > 15:
                last = time.time(); el = time.time() - t0
                print(f"      {len(done)}/{n_seg} segments, {got['bytes']/1e9:.2f} GB this run, "
                      f"{got['bytes']/1e6/el:.0f} MB/s")
    os.close(fd); os.remove(prog)
    print(f"      done in {time.time()-t0:.0f} s ({got['bytes']/1e6/max(time.time()-t0,1e-6):.0f} MB/s)")
    return path


import hashlib, os, struct, time, zlib, urllib.request, urllib.error

_LOCAL_SIG, _CENTRAL_SIG, _DESC_SIG = 0x04034B50, 0x02014B50, 0x08074B50


def http_part_stream(urls_md5, chunk=1 << 22, max_retries=20, _test_fail_after=None):
    """Yield the concatenated bytes of the parts. Each part is verified against its md5 when it
    ends. On any network error the part is re-requested from the byte reached (Range)."""
    for url, md5 in urls_md5:
        have, h, attempt, failed_once = 0, hashlib.md5(), 0, False
        print(f"    streaming {url.split('/')[-1].split('?')[0]}")
        while True:
            req = urllib.request.Request(url, headers={"User-Agent": "conformalorb/1.0"})
            if have:
                req.add_header("Range", f"bytes={have}-")
            try:
                with urllib.request.urlopen(req, timeout=60) as resp:
                    if have and resp.getcode() != 206:
                        raise RuntimeError("server does not support Range resume")
                    while True:
                        buf = resp.read(chunk)
                        if not buf:
                            break
                        if _test_fail_after is not None and not failed_once and have + len(buf) > _test_fail_after:
                            failed_once = True
                            raise ConnectionResetError("simulated drop")
                        have += len(buf); h.update(buf)
                        yield buf
                break                                    # part complete
            except (urllib.error.URLError, ConnectionError, TimeoutError, OSError) as e:
                attempt += 1
                if attempt > max_retries:
                    raise
                print(f"      {type(e).__name__} at {have/1e9:.2f} GB; resuming (attempt {attempt})")
                time.sleep(min(60, 5 * attempt))
        got = h.hexdigest()
        if md5 and got != md5:
            raise RuntimeError(f"md5 mismatch for part: {got} != {md5}")
        print(f"      part complete: {have/1e9:.2f} GB, md5 {'ok' if md5 else 'not checked'}")


class ByteReader:
    """Blocking read(n) over a chunk generator, with push-back."""
    def __init__(self, gen):
        self.gen, self.buf, self.eof, self.total = gen, bytearray(), False, 0
    def _fill(self, n):
        while len(self.buf) < n and not self.eof:
            try:
                self.buf += next(self.gen)
            except StopIteration:
                self.eof = True
    def read(self, n):
        self._fill(n); out = bytes(self.buf[:n]); del self.buf[:n]; self.total += len(out); return out
    def read_exact(self, n):
        out = self.read(n)
        if len(out) != n:
            raise EOFError("stream ended early")
        return out
    def push_back(self, data):
        self.buf[:0] = data; self.total -= len(data)


def _parse_zip64_sizes(extra, comp, uncomp):
    """Return (comp, uncomp, has_zip64) with zip64 values substituted where the 32-bit fields are 0xFFFFFFFF.
    has_zip64 also decides the width of a data descriptor's size fields (APPNOTE 4.3.9.2)."""
    i, has_zip64 = 0, False
    while i + 4 <= len(extra):
        tag, size = struct.unpack("<HH", extra[i:i + 4]); body = extra[i + 4:i + 4 + size]
        if tag == 0x0001:
            has_zip64 = True
            vals = list(struct.unpack("<" + "Q" * (len(body) // 8), body[:len(body) // 8 * 8]))
            if uncomp == 0xFFFFFFFF and vals: uncomp = vals.pop(0)
            if comp == 0xFFFFFFFF and vals: comp = vals.pop(0)
        i += 4 + size
    return comp, uncomp, has_zip64


def stream_extract_zip(reader, dest_dir, want_suffix=".db", chunk=1 << 22):
    """Walk local headers; write members ending with `want_suffix` to dest_dir. Returns list of paths."""
    os.makedirs(dest_dir, exist_ok=True); written = []
    while True:
        sig_bytes = reader.read(4)
        if len(sig_bytes) < 4:
            break
        sig, = struct.unpack("<I", sig_bytes)
        if sig != _LOCAL_SIG:                       # central directory reached: every member has been read
            break
        (ver, flags, method, _t, _d, crc, comp, uncomp, nlen, xlen) = struct.unpack("<HHHHHIIIHH", reader.read_exact(26))
        name = reader.read_exact(nlen).decode("utf-8", "replace"); extra = reader.read_exact(xlen)
        comp, uncomp, has_zip64 = _parse_zip64_sizes(extra, comp, uncomp)
        has_desc = bool(flags & 0x0008)
        if method not in (0, 8):
            raise RuntimeError(f"unsupported zip compression method {method} for {name} "
                               f"(9 = Deflate64: extract the parts with 7-Zip instead)")
        if name.endswith("/"):
            continue
        keep = name.endswith(want_suffix)
        out_path = os.path.join(dest_dir, os.path.basename(name)) if keep else None
        fh = open(out_path, "wb") if keep else None
        print(f"    member {name}: method={'deflate' if method == 8 else 'stored'}, "
              f"sizes {'in data descriptor' if has_desc and comp in (0, 0xFFFFFFFF) else f'{uncomp/1e9:.2f} GB'}"
              f"{' -> extracting' if keep else ' -> skipped'}")
        got_crc, n_out, t0, last = 0, 0, time.time(), 0

        def emit(data):
            nonlocal got_crc, n_out, last
            if fh:
                fh.write(data); got_crc = zlib.crc32(data, got_crc)
            n_out += len(data)
            if n_out - last > 2e9:
                last = n_out; print(f"      {n_out/1e9:.1f} GB written ({n_out/1e6/max(time.time()-t0,1e-6):.0f} MB/s)")

        if method == 0:                                # stored: sizes must be known
            if has_desc and comp in (0, 0xFFFFFFFF):
                raise RuntimeError("stored member with unknown size cannot be streamed")
            left = comp
            while left:
                b = reader.read_exact(min(chunk, left)); emit(b); left -= len(b)
        else:                                          # deflate
            d = zlib.decompressobj(-15)
            if has_desc and comp in (0, 0xFFFFFFFF):   # size unknown: run until the deflate stream ends
                while not d.eof:
                    b = reader.read(chunk)
                    if not b: raise EOFError("stream ended inside a deflate member")
                    emit(d.decompress(b))
                reader.push_back(d.unused_data)
            else:
                left = comp
                while left:
                    b = reader.read_exact(min(chunk, left)); emit(d.decompress(b)); left -= len(b)
                emit(d.flush())
        if fh:
            fh.close()
        if has_desc:                                   # data descriptor: [sig] crc sizes (4+4, or 8+8 when zip64)
            head = reader.read_exact(4)
            if struct.unpack("<I", head)[0] == _DESC_SIG:
                head = reader.read_exact(4)
            crc = struct.unpack("<I", head)[0]
            reader.read_exact(16 if has_zip64 else 8)
        if keep:
            if got_crc != crc:
                raise RuntimeError(f"CRC mismatch for {name}: {got_crc:08x} != {crc:08x}")
            print(f"      {name}: {n_out/1e9:.2f} GB written, CRC ok, {time.time()-t0:.0f} s")
            written.append(out_path)
    return written


def get_qh9_database_streaming(data_dir, urls_md5, min_free_gb=35, _test_fail_after=None):
    os.makedirs(data_dir, exist_ok=True)
    import shutil
    free = shutil.disk_usage(data_dir).free / 1e9
    print(f"[3s] streaming extraction into {data_dir}; free disk {free:.0f} GB (need ~{min_free_gb} GB)")
    if free < min_free_gb:
        print("    NOT ENOUGH DISK for the database itself."); return None
    reader = ByteReader(http_part_stream(urls_md5, _test_fail_after=_test_fail_after))
    paths = stream_extract_zip(reader, data_dir)
    db = next((p for p in paths if os.path.basename(p) == "QH9Stable.db"), paths[0] if paths else None)
    if db:
        print(f"    ready: {db} ({os.path.getsize(db)/1e9:.2f} GB)")
    return db


Writing co_acquire.py


In [4]:
%%writefile co_subset.py
# ConformalOrb: official QH9 splits + compact subset database
import numpy as np, sqlite3, os, time

def official_splits(conn):
    ids = np.array([r[0] for r in conn.execute("SELECT id FROM data ORDER BY id")])
    Ns  = np.array([r[0] for r in conn.execute("SELECT N FROM data ORDER BY id")])
    n = len(ids)
    if not np.array_equal(ids, np.arange(n)):
        print("  note: ids are not contiguous 0..N-1; split masks are mapped by row position")
    # random split (QH9-stable-id)
    perm = np.random.RandomState(seed=43).permutation(n)
    n_tr, n_va = int(n * 0.8), int(n * 0.1)
    rnd = dict(train=ids[perm[:n_tr]], val=ids[perm[n_tr:n_tr + n_va]], test=ids[perm[n_tr + n_va:]])
    # size split (QH9-stable-ood)
    ood = dict(train=ids[Ns <= 20], val=ids[(Ns >= 21) & (Ns <= 22)], test=ids[Ns >= 23])
    return rnd, ood, dict(zip(ids.tolist(), Ns.tolist()))

def pack_upper(F):
    n = F.shape[0]; iu = np.triu_indices(n)
    return np.ascontiguousarray(F[iu].astype(np.float64))

def unpack_upper(packed, n):
    F = np.zeros((n, n)); iu = np.triu_indices(n); F[iu] = packed
    return F + F.T - np.diag(np.diag(F))

def build_subset_db(conn_full, out_path, id_test_ids, ood_test_ids, rng, n_id=0, n_ood=0):
    """Write a compact DB holding a random subsample of the official id-test and ood-test molecules.
    The `meta` table records provenance and the packing so nothing is ambiguous later."""
    n_id  = len(id_test_ids)  if n_id  <= 0 else min(n_id,  len(id_test_ids))
    n_ood = len(ood_test_ids) if n_ood <= 0 else min(n_ood, len(ood_test_ids))
    sel_id  = np.sort(rng.choice(id_test_ids,  size=n_id,  replace=False))
    sel_ood = np.sort(rng.choice(ood_test_ids, size=n_ood, replace=False))
    if os.path.exists(out_path): os.remove(out_path)
    out = sqlite3.connect(out_path); cur = out.cursor()
    cur.execute("CREATE TABLE data (id INTEGER NOT NULL PRIMARY KEY, N INTEGER, Z BLOB, pos BLOB, Ham BLOB)")
    cur.execute("CREATE TABLE meta (key TEXT PRIMARY KEY, value TEXT)")
    cur.execute("CREATE TABLE membership (id INTEGER, split TEXT, PRIMARY KEY (id, split))")
    cur.executemany("INSERT INTO meta VALUES (?,?)", [
        ("source", "QH9Stable.db (Zenodo 8274793, B3LYP/def2-SVP, PySCF AO order)"),
        ("ham_packing", "upper_triangle_float64_rowmajor_including_diagonal"),
        ("n_orbitals_rule", "5 per H/He, 14 per C/N/O/F (spherical def2-SVP)"),
        ("id_test_subsample", str(len(sel_id))), ("ood_test_subsample", str(len(sel_ood))),
        ("built", time.strftime("%Y-%m-%d"))])
    # The two official splits are different partitions of the SAME 130,831 molecules, so the id-test
    # and ood-test sets overlap (about 10% of ood-test). Each molecule is stored once; `membership`
    # records every split it belongs to.
    t0 = time.time(); n_bytes = 0; stored = set()
    for split, sel in (("id_test", sel_id), ("ood_test", sel_ood)):
        for j, i in enumerate(sel):
            i = int(i)
            if i not in stored:
                N, Zb, posb, Hb = conn_full.execute("SELECT N, Z, pos, Ham FROM data WHERE id=?", (i,)).fetchone()
                n_orb = int(round(np.sqrt(len(Hb) / 8)))
                F = np.frombuffer(Hb, np.float64).reshape(n_orb, n_orb)
                packed = pack_upper(0.5 * (F + F.T))
                cur.execute("INSERT INTO data VALUES (?,?,?,?,?)", (i, int(N), Zb, posb, memoryview(packed)))
                n_bytes += packed.nbytes + len(Zb) + len(posb); stored.add(i)
            cur.execute("INSERT OR IGNORE INTO membership VALUES (?,?)", (i, split))
            if (j + 1) % 1000 == 0:
                out.commit(); print(f"  {split}: {j+1}/{len(sel)}  ({n_bytes/1e9:.2f} GB so far, {time.time()-t0:.0f} s)")
    overlap = len(set(map(int, sel_id)) & set(map(int, sel_ood)))
    cur.execute("INSERT INTO meta VALUES (?,?)", ("id_test_ood_test_overlap", str(overlap)))
    out.commit(); out.close()
    print(f"subset written: {out_path}  {os.path.getsize(out_path)/1e9:.2f} GB  "
          f"({len(sel_id)} id-test + {len(sel_ood)} ood-test, {overlap} in both -> {len(stored)} distinct molecules)")
    return out_path


Writing co_subset.py


In [5]:
%%writefile co_phase0.py
#!/usr/bin/env python3
"""
ConformalOrb -- Phase 0 verification
====================================

Purpose
-------
Before spending a single GPU-hour on training a Hamiltonian model, verify
numerically that the two mathematical objects the paper rests on actually
behave as claimed:

  TEST 1  Gauge invariance.
          The proposed nonconformity score s_B (normalised chordal Grassmann
          distance between frontier subspaces) is invariant to sign flips,
          to arbitrary orthogonal rotations inside the subspace, and to
          permutations of degenerate orbitals -- while the naive per-orbital
          score 1 - |<psi_ref|psi_pert>|^2 is not.

  TEST 2  Degeneracy.
          On a molecule with a genuinely degenerate HOMO (benzene, e1g pair)
          an arbitrary rotation inside the degenerate pair is a physically
          equivalent SCF solution. s_B must return exactly 0; the naive
          score must not.

  TEST 3  Perturbation bounds.
          For random symmetric perturbations dF of the Fock matrix, the
          Davis-Kahan sin(theta) bound and the Weyl eigenvalue bound must
          hold in 100% of trials, and we measure how tight they are.
          We report BOTH the sharp form (using ||X^T dF X||_2 directly) and
          the loose form (||dF||_2 / lambda_min(S)) to quantify the price of
          the ill-conditioning of the AO overlap matrix S.

Level of theory is B3LYP/def2-SVP -- deliberately identical to the QH9
dataset -- so every number produced here transfers directly to QH9 later.

Usage
-----
    pip install pyscf numpy
    python conformalorb_phase0.py                    # default molecule set
    python conformalorb_phase0.py --trials 100       # more DK trials
    python conformalorb_phase0.py --outdir results/

Outputs
-------
    phase0_summary.csv        one row per molecule (lambda_min(S), gaps, ...)
    phase0_gauge.csv          gauge-invariance test results
    phase0_davis_kahan.csv    one row per (molecule, noise level, trial)

Author: A. A. Khairbek
"""

from __future__ import annotations

import argparse
import os
import sys
import csv
import math
import numpy as np

try:
    from pyscf import gto, dft
except ImportError:
    sys.exit("PySCF is required:  pip install pyscf")


# --------------------------------------------------------------------------
# Molecule set: QM9-like (C, N, O, F, H only; <= 9 heavy atoms).
# Benzene and cyclopropane are included specifically because they carry
# genuinely degenerate frontier orbitals.
# --------------------------------------------------------------------------
MOLECULES = {
    "water":        "O 0.0000 0.0000 0.1173; H 0.0000 0.7572 -0.4692; "
                    "H 0.0000 -0.7572 -0.4692",
    "methanol":     "C -0.0470 0.6635 0.0000; O -0.0470 -0.7585 0.0000; "
                    "H -1.0740 1.0106 0.0000; H 0.4880 1.0429 0.8790; "
                    "H 0.4880 1.0429 -0.8790; H 0.8620 -1.0407 0.0000",
    "ethylene":     "C 0.0000 0.0000 0.6695; C 0.0000 0.0000 -0.6695; "
                    "H 0.0000 0.9289 1.2321; H 0.0000 -0.9289 1.2321; "
                    "H 0.0000 0.9289 -1.2321; H 0.0000 -0.9289 -1.2321",
    "formamide":    "C 0.0000 0.4380 0.0000; O 1.1650 0.7600 0.0000; "
                    "N -1.0180 1.2870 0.0000; H -0.2200 -0.6250 0.0000; "
                    "H -1.9640 0.9700 0.0000; H -0.8380 2.2760 0.0000",
    "acetonitrile": "C 0.0000 0.0000 1.1860; N 0.0000 0.0000 -0.0130; "
                    "C 0.0000 0.0000 2.6460; H 0.0000 1.0270 3.0140; "
                    "H 0.8894 -0.5135 3.0140; H -0.8894 -0.5135 3.0140",
    "cyclopropane": "C 0.0000 0.8698 0.0000; C 0.7533 -0.4349 0.0000; "
                    "C -0.7533 -0.4349 0.0000; H 0.0000 1.4576 0.9179; "
                    "H 0.0000 1.4576 -0.9179; H 1.2624 -0.7288 0.9179; "
                    "H 1.2624 -0.7288 -0.9179; H -1.2624 -0.7288 0.9179; "
                    "H -1.2624 -0.7288 -0.9179",
    "pyridine":     "N 0.0000 0.0000 1.4160; C 0.0000 1.1400 0.7150; "
                    "C 0.0000 -1.1400 0.7150; C 0.0000 1.1950 -0.6750; "
                    "C 0.0000 -1.1950 -0.6750; C 0.0000 0.0000 -1.3800; "
                    "H 0.0000 2.0430 1.3130; H 0.0000 -2.0430 1.3130; "
                    "H 0.0000 2.1420 -1.1990; H 0.0000 -2.1420 -1.1990; "
                    "H 0.0000 0.0000 -2.4640",
    "benzene":      "C 0.0000 1.3970 0.0000; C 1.2098 0.6985 0.0000; "
                    "C 1.2098 -0.6985 0.0000; C 0.0000 -1.3970 0.0000; "
                    "C -1.2098 -0.6985 0.0000; C -1.2098 0.6985 0.0000; "
                    "H 0.0000 2.4810 0.0000; H 2.1486 1.2405 0.0000; "
                    "H 2.1486 -1.2405 0.0000; H 0.0000 -2.4810 0.0000; "
                    "H -2.1486 -1.2405 0.0000; H -2.1486 1.2405 0.0000",
}

HARTREE_TO_EV = 27.211386245988


# ==========================================================================
# Core linear algebra
# ==========================================================================

def orthogonalizer(S: np.ndarray, thresh: float = 1e-7):
    """Canonical orthogonalisation X such that X^T S X = I.

    Returns (X, s_eigvals, n_dropped).  Eigenvalues of S below `thresh`
    are dropped -- this is what keeps the Weyl/Davis-Kahan bounds from
    blowing up on near-linearly-dependent AO basis sets.
    """
    s_eig, U = np.linalg.eigh(S)
    keep = s_eig > thresh
    n_dropped = int((~keep).sum())
    X = U[:, keep] / np.sqrt(s_eig[keep])
    return X, s_eig, n_dropped


def solve_generalized(F: np.ndarray, S: np.ndarray, X: np.ndarray):
    """Solve F C = S C eps via the orthogonalised standard problem.

    Returns (eps, C) with C^T S C = I, eigenvalues ascending.
    """
    F_tilde = X.T @ F @ X
    F_tilde = 0.5 * (F_tilde + F_tilde.T)          # enforce symmetry
    eps, C_tilde = np.linalg.eigh(F_tilde)
    C = X @ C_tilde
    return eps, C


def frontier_window(eps: np.ndarray, n_occ: int,
                    n_below: int = 2, n_above: int = 2,
                    degen_tol_ev: float = 0.05):
    """Degeneracy-aware frontier window.

    Starts from [HOMO-(n_below-1) ... LUMO+(n_above-1)] and then *expands*
    each boundary until it no longer cuts through a (near-)degenerate
    cluster.  Splitting a degenerate cluster is precisely what makes the
    naive per-orbital score ill-defined, so the window must never do it.

    Returns (i_lo, i_hi) inclusive indices into `eps`.
    """
    tol = degen_tol_ev / HARTREE_TO_EV
    i_lo = max(0, n_occ - n_below)
    i_hi = min(len(eps) - 1, n_occ + n_above - 1)

    while i_lo > 0 and (eps[i_lo] - eps[i_lo - 1]) < tol:
        i_lo -= 1
    while i_hi < len(eps) - 1 and (eps[i_hi + 1] - eps[i_hi]) < tol:
        i_hi += 1
    return i_lo, i_hi


def principal_cosines(C_ref: np.ndarray, C_pert: np.ndarray,
                      S: np.ndarray, idx: slice) -> np.ndarray:
    """Cosines of the principal angles between two frontier subspaces.

    Because C^T S C = I, the overlap matrix M = C_ref^T S C_pert has
    singular values sigma_i = cos(theta_i) in [0, 1].
    """
    M = C_ref[:, idx].T @ S @ C_pert[:, idx]
    sigma = np.linalg.svd(M, compute_uv=False)
    return np.clip(sigma, 0.0, 1.0)


def score_subspace(C_ref, C_pert, S, idx) -> float:
    """PROPOSED SCORE  s_B  -- normalised chordal Grassmann distance.

        s_B = sqrt( 1 - (1/k) * sum_i cos^2(theta_i) )

    In [0, 1].  Invariant under: sign flips, any orthogonal rotation
    inside the subspace, and permutation of degenerate orbitals.
    """
    sigma = principal_cosines(C_ref, C_pert, S, idx)
    k = len(sigma)
    return math.sqrt(max(0.0, 1.0 - np.sum(sigma ** 2) / k))


def sin_theta_max(C_ref, C_pert, S, idx) -> float:
    """Largest principal angle, sin(theta_max) -- the Davis-Kahan quantity."""
    sigma = principal_cosines(C_ref, C_pert, S, idx)
    return math.sqrt(max(0.0, 1.0 - sigma.min() ** 2))


def score_naive(C_ref, C_pert, S, i: int) -> float:
    """BASELINE SCORE -- the per-orbital overlap score from the original
    project sketch:   1 - |<psi_ref | psi_pert>|^2   for a single orbital i.
    Included only to demonstrate that it fails under degeneracy.
    """
    ov = float(C_ref[:, i].T @ S @ C_pert[:, i])
    return 1.0 - ov ** 2


def random_orthogonal(k: int, rng) -> np.ndarray:
    """Haar-random orthogonal k x k matrix."""
    A = rng.standard_normal((k, k))
    Q, R = np.linalg.qr(A)
    return Q * np.sign(np.diag(R))


def random_symmetric(n: int, spectral_norm: float, rng) -> np.ndarray:
    """Random symmetric perturbation with prescribed spectral norm."""
    G = rng.standard_normal((n, n))
    E = 0.5 * (G + G.T)
    nrm = np.linalg.norm(E, 2)
    return E * (spectral_norm / nrm)


# ==========================================================================
# Reference data
# ==========================================================================

def run_scf(name: str, atom: str, basis="def2-svp", xc="b3lyp5", verbose=0):
    """Converged B3LYP/def2-SVP reference -- same level of theory as QH9."""
    mol = gto.M(atom=atom, basis=basis, unit="Angstrom", verbose=verbose)
    mf = dft.RKS(mol)
    mf.xc = xc
    mf.grids.level = 3
    mf.conv_tol = 1e-12
    mf.kernel()
    if not mf.converged:
        raise RuntimeError(f"SCF did not converge for {name}")

    S = mf.get_ovlp()
    dm = mf.make_rdm1()
    F = mf.get_fock(dm=dm)                       # converged Kohn-Sham matrix
    F = 0.5 * (F + F.T)
    n_occ = mol.nelectron // 2
    return dict(name=name, mol=mol, S=S, F=F, n_occ=n_occ,
                n_basis=S.shape[0], e_tot=mf.e_tot)


# ==========================================================================
# TEST 1 + 2 -- gauge invariance and degeneracy
# ==========================================================================

def test_gauge(ref: dict, rng, degen_tol_ev: float):
    S, F = ref["S"], ref["F"]
    X, s_eig, n_drop = orthogonalizer(S)
    eps, C = solve_generalized(F, S, X)
    n_occ = ref["n_occ"]

    i_lo, i_hi = frontier_window(eps, n_occ, degen_tol_ev=degen_tol_ev)
    idx = slice(i_lo, i_hi + 1)
    k = i_hi - i_lo + 1

    homo_gap_ev = (eps[n_occ] - eps[n_occ - 1]) * HARTREE_TO_EV
    homo_degen_ev = (eps[n_occ - 1] - eps[n_occ - 2]) * HARTREE_TO_EV

    rows = []

    def record(op, C2, note=""):
        sb = score_subspace(C, C2, S, idx)
        nv = score_naive(C, C2, S, n_occ - 1)          # HOMO
        rows.append(dict(molecule=ref["name"], operation=op,
                         k=k, s_B=sb, s_naive_HOMO=nv, note=note))

    # (a) identity -- sanity
    record("identity", C.copy())

    # (b) sign flip of every frontier orbital
    C2 = C.copy()
    C2[:, idx] *= -1.0
    record("sign_flip_all", C2)

    # (c) random sign pattern
    C2 = C.copy()
    signs = rng.choice([-1.0, 1.0], size=k)
    C2[:, idx] = C2[:, idx] * signs
    record("random_sign_pattern", C2)

    # (d) Haar-random orthogonal rotation inside the whole frontier subspace
    C2 = C.copy()
    Q = random_orthogonal(k, rng)
    C2[:, idx] = C[:, idx] @ Q
    record("random_rotation_in_subspace", C2)

    # (e) permutation of the frontier orbitals
    C2 = C.copy()
    perm = rng.permutation(k)
    C2[:, idx] = C[:, idx][:, perm]
    record("permutation", C2)

    # (f) THE DEGENERACY TEST -- rotate only inside the degenerate HOMO pair.
    #     If |eps_HOMO - eps_HOMO-1| is below tolerance, EVERY angle below
    #     gives a physically equivalent SCF solution.  A valid score must
    #     return 0 for all of them.  We sweep the angle so the failure of the
    #     naive score is visible as a full sin^2(theta) curve: the same pair
    #     of physically identical wavefunctions can be scored anywhere from
    #     0 to 1 depending on nothing but the diagonaliser's arbitrary choice.
    if abs(homo_degen_ev) < degen_tol_ev:
        pair = [n_occ - 2, n_occ - 1]
        for deg in (15, 30, 45, 60, 90):
            theta = math.radians(deg)
            c, s_ = math.cos(theta), math.sin(theta)
            C2 = C.copy()
            C2[:, pair] = C[:, pair] @ np.array([[c, -s_], [s_, c]])
            record(f"rotate_degenerate_HOMO_pair_{deg}deg", C2,
                   note=f"pair split = {homo_degen_ev:.4f} eV")

    summary = dict(molecule=ref["name"], n_basis=ref["n_basis"],
                   n_occ=n_occ, k=k, window=f"{i_lo}-{i_hi}",
                   lambda_min_S=float(s_eig.min()),
                   cond_S=float(s_eig.max() / s_eig.min()),
                   n_dropped=n_drop,
                   homo_lumo_gap_eV=homo_gap_ev,
                   homo_pair_split_eV=homo_degen_ev,
                   e_tot=ref["e_tot"])
    return rows, summary, (eps, C, X, idx, i_lo, i_hi)


# ==========================================================================
# TEST 3 -- Davis-Kahan and Weyl bounds
# ==========================================================================

def test_perturbation(ref, state, noise_levels, n_trials, rng):
    S, F = ref["S"], ref["F"]
    eps, C, X, idx, i_lo, i_hi = state
    n = F.shape[0]
    lam_min_S = float(np.linalg.eigvalsh(S).min())

    # Unperturbed spectral gap separating the frontier block from the rest,
    # measured in the ORTHOGONALISED problem (which is where DK applies).
    gap_lo = eps[i_lo] - eps[i_lo - 1] if i_lo > 0 else np.inf
    gap_hi = eps[i_hi + 1] - eps[i_hi] if i_hi < len(eps) - 1 else np.inf
    delta0 = min(gap_lo, gap_hi)

    rows = []
    for nrm in noise_levels:
        for t in range(n_trials):
            dF = random_symmetric(n, nrm, rng)
            dF_t = X.T @ dF @ X                       # perturbation actually
            dF_t = 0.5 * (dF_t + dF_t.T)              # seen by the eigenproblem
            nrm_sharp = float(np.linalg.norm(dF_t, 2))
            nrm_loose = nrm / lam_min_S               # worst-case surrogate

            eps_p, C_p = solve_generalized(F + dF, S, X)

            # --- observed quantities
            sin_obs = sin_theta_max(C, C_p, S, idx)
            d_eps_obs = float(np.max(np.abs(eps_p[idx] - eps[idx])))

            # --- Weyl bound on orbital energies
            weyl_sharp = nrm_sharp
            weyl_loose = nrm_loose

            # --- Davis-Kahan sin(theta) bound; the effective gap must be
            #     shrunk by ||E|| because the *perturbed* spectrum is what
            #     has to stay outside the interval (Weyl).
            delta_sharp = delta0 - nrm_sharp
            delta_loose = delta0 - nrm_loose
            dk_sharp = nrm_sharp / delta_sharp if delta_sharp > 0 else np.inf
            dk_loose = nrm_loose / delta_loose if delta_loose > 0 else np.inf

            rows.append(dict(
                molecule=ref["name"], noise_spectral_norm=nrm, trial=t,
                lambda_min_S=lam_min_S,
                dF_tilde_norm=nrm_sharp,
                dF_loose_norm=nrm_loose,
                loose_over_sharp=nrm_loose / nrm_sharp,
                delta0_Ha=float(delta0),
                sin_theta_obs=sin_obs,
                dk_bound_sharp=float(dk_sharp),
                dk_bound_loose=float(dk_loose),
                dk_sharp_holds=bool(sin_obs <= dk_sharp + 1e-12),
                dk_sharp_tightness=float(dk_sharp / sin_obs) if sin_obs > 0 else np.nan,
                deps_obs_Ha=d_eps_obs,
                weyl_sharp=weyl_sharp,
                weyl_loose=weyl_loose,
                weyl_sharp_holds=bool(d_eps_obs <= weyl_sharp + 1e-12),
                weyl_sharp_tightness=float(weyl_sharp / d_eps_obs) if d_eps_obs > 0 else np.nan,
                s_B=score_subspace(C, C_p, S, idx),
            ))
    return rows


# ==========================================================================
# Driver
# ==========================================================================

def write_csv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def main():
    p = argparse.ArgumentParser(description="ConformalOrb Phase 0 verification")
    p.add_argument("--molecules", nargs="*", default=list(MOLECULES),
                   help="subset of molecules to run")
    p.add_argument("--trials", type=int, default=20,
                   help="Davis-Kahan trials per noise level")
    p.add_argument("--degen-tol", type=float, default=0.05,
                   help="degeneracy tolerance in eV for window expansion")
    p.add_argument("--seed", type=int, default=20260903)
    p.add_argument("--outdir", default=".")
    args = p.parse_args()

    os.makedirs(args.outdir, exist_ok=True)
    rng = np.random.default_rng(args.seed)
    noise_levels = [1e-5, 1e-4, 1e-3, 1e-2]        # Hartree, spectral norm

    all_gauge, all_summary, all_dk = [], [], []

    for name in args.molecules:
        print(f"[SCF ] {name} ...", flush=True)
        ref = run_scf(name, MOLECULES[name])
        g_rows, summ, state = test_gauge(ref, rng, args.degen_tol)
        dk_rows = test_perturbation(ref, state, noise_levels, args.trials, rng)
        all_gauge += g_rows
        all_summary.append(summ)
        all_dk += dk_rows

    write_csv(os.path.join(args.outdir, "phase0_summary.csv"), all_summary)
    write_csv(os.path.join(args.outdir, "phase0_gauge.csv"), all_gauge)
    write_csv(os.path.join(args.outdir, "phase0_davis_kahan.csv"), all_dk)

    # ---------------------------------------------------------------- report
    print("\n" + "=" * 78)
    print("TEST 1/2 -- GAUGE INVARIANCE")
    print("=" * 78)
    print(f"{'molecule':<14}{'operation':<32}{'s_B':>12}{'s_naive(HOMO)':>16}")
    print("-" * 78)
    for r in all_gauge:
        print(f"{r['molecule']:<14}{r['operation']:<32}"
              f"{r['s_B']:>12.2e}{r['s_naive_HOMO']:>16.2e}")

    # s_B = sqrt(1 - mean(sigma^2)).  Taking a square root of a quantity that
    # is zero to within machine epsilon means the numerical resolution floor
    # of this score is sqrt(eps) ~ 1.5e-8, NOT eps.  Scores below this floor
    # are indistinguishable from an exact match -- which matters when the
    # score is later used for conformal calibration.
    floor = math.sqrt(np.finfo(float).eps)
    max_sb = max(r["s_B"] for r in all_gauge)
    degen_rows = [r for r in all_gauge
                  if r["operation"].startswith("rotate_degenerate")]
    print("-" * 78)
    print(f"resolution floor sqrt(machine eps) : {floor:.3e}")
    print(f"max s_B over ALL gauge operations  : {max_sb:.3e}   "
          f"-> {'PASS' if max_sb < 10 * floor else 'FAIL'}")
    if degen_rows:
        lo = min(r["s_naive_HOMO"] for r in degen_rows)
        hi = max(r["s_naive_HOMO"] for r in degen_rows)
        print(f"degenerate-pair rotation: s_naive spans [{lo:.3f}, {hi:.3f}] "
              f"across physically IDENTICAL solutions, while s_B stays at 0")
        print("   -> the per-orbital overlap score is ill-defined; s_B is not.")

    print("\n" + "=" * 78)
    print("TEST 3 -- PERTURBATION BOUNDS")
    print("=" * 78)
    dk_ok = all(r["dk_sharp_holds"] for r in all_dk if np.isfinite(r["dk_bound_sharp"]))
    wy_ok = all(r["weyl_sharp_holds"] for r in all_dk)
    n_inf = sum(1 for r in all_dk if not np.isfinite(r["dk_bound_sharp"]))
    print(f"Weyl  bound held in {sum(r['weyl_sharp_holds'] for r in all_dk)}"
          f"/{len(all_dk)} trials -> {'PASS' if wy_ok else 'FAIL'}")
    print(f"Davis-Kahan held in "
          f"{sum(r['dk_sharp_holds'] for r in all_dk)}/{len(all_dk)} trials"
          f" -> {'PASS' if dk_ok else 'FAIL'}   ({n_inf} vacuous: ||E|| > gap)")

    print("\nTightness (bound / observed), median by noise level:")
    print(f"{'||dF||_2':>10}{'DK':>12}{'Weyl':>12}{'loose/sharp':>14}")
    for nrm in noise_levels:
        sub = [r for r in all_dk if r["noise_spectral_norm"] == nrm]
        dk = np.median([r["dk_sharp_tightness"] for r in sub
                        if np.isfinite(r["dk_sharp_tightness"])])
        wy = np.median([r["weyl_sharp_tightness"] for r in sub])
        lo = np.median([r["loose_over_sharp"] for r in sub])
        print(f"{nrm:>10.0e}{dk:>12.1f}{wy:>12.1f}{lo:>14.1f}")

    print("\n" + "=" * 78)
    print("OVERLAP MATRIX CONDITIONING (the price of the loose bound)")
    print("=" * 78)
    print(f"{'molecule':<14}{'n_basis':>9}{'lambda_min(S)':>16}"
          f"{'cond(S)':>12}{'gap(eV)':>10}{'split(eV)':>11}{'k':>4}")
    print("-" * 78)
    for s in all_summary:
        print(f"{s['molecule']:<14}{s['n_basis']:>9}{s['lambda_min_S']:>16.3e}"
              f"{s['cond_S']:>12.1f}{s['homo_lumo_gap_eV']:>10.2f}"
              f"{s['homo_pair_split_eV']:>11.3f}{s['k']:>4}")

    print(f"\nCSV written to {os.path.abspath(args.outdir)}")


if __name__ == "__main__":
    main()


Writing co_phase0.py


In [6]:
%%writefile co_survey.py
#!/usr/bin/env python3
"""
ConformalOrb -- Phase 1a: survey of the real QH9-stable dataset
================================================================

What this answers (on 20k real molecules instead of 8 hand-picked ones):

  Q1  How ill-conditioned is the def2-SVP overlap matrix S across QH9?
      -> decides whether the LOOSE analytic certificate ||dF||_2 / lambda_min(S)
         is usable at all, or whether we must conformalise ||X^T dF X||_2
         directly in the orthogonalised basis.

  Q2  How often are frontier orbitals (near-)degenerate?
      -> the empirical justification for the subspace score s_B.

  Q3  How large is the spectral gap delta0 around the frontier block?
      -> a Davis-Kahan certificate is VACUOUS whenever ||dF_tilde||_2 > delta0.
         Comparing delta0 to realistic ML Hamiltonian errors tells us, molecule
         by molecule, where the analytic route can and cannot speak.

  Q4  What is the HOMO-LUMO gap distribution?
      -> Mondrian bin edges for gap-conditional coverage.

Storage format of QH9Stable.db (verified against the official generation and
loader code in divelab/AIRS, OpenDFT/QHBench/QH9):

    SQLite table `data`:  id INTEGER, N INTEGER, Z BLOB, pos BLOB, Ham BLOB
      Z    : int32 (or int64) atomic numbers, length N
      pos  : float64, shape (N, 3), Angstrom
      Ham  : float64, shape (n_orb, n_orb) -- the CONVERGED KOHN-SHAM MATRIX
             (mf.get_fock() after RKS/B3LYP/def2-SVP), in PySCF AO order,
             spherical basis (H: 5 functions, C/N/O/F: 14 functions).
    The overlap matrix S is NOT stored; it is analytic and is recomputed
    here with PySCF from the geometry -- this is exactly what the official
    QHNet evaluation code does as well.

Usage
-----
    python qh9_survey.py --db /path/to/QH9Stable.db --n-sample 20000
    python qh9_survey.py --db QH9Stable.db --n-sample 2000 --check-scf 5 --plot

Requires: numpy, pyscf  (matplotlib optional for --plot)

Author: A. A. Khairbek
"""

from __future__ import annotations

import argparse
import csv
import math
import os
import sqlite3
import sys
import time

import numpy as np

try:
    from pyscf import gto, dft
except ImportError:
    sys.exit("PySCF is required:  pip install pyscf")

HARTREE_TO_EV = 27.211386245988
QH9_BASIS = "def2svp"          # exactly as in the official generation script
QH9_XC = "b3lyp5"   # QH9-stable = PySCF 2.2.1 "b3lyp" = VWN5 variant


# ==========================================================================
# Reading the raw database
# ==========================================================================

def _decode_int(buf: bytes, n: int) -> np.ndarray:
    """Atomic numbers blob -> int array; dtype inferred from the byte length
    so the reader works whether the writer used int32 or int64."""
    if len(buf) == 4 * n:
        return np.frombuffer(buf, np.int32).astype(int)
    if len(buf) == 8 * n:
        return np.frombuffer(buf, np.int64).astype(int)
    raise ValueError(f"Z blob has {len(buf)} bytes for N={n}")


def _decode_float(buf: bytes, n_elem: int) -> np.ndarray:
    if len(buf) == 8 * n_elem:
        return np.frombuffer(buf, np.float64)
    if len(buf) == 4 * n_elem:
        return np.frombuffer(buf, np.float32).astype(np.float64)
    raise ValueError(f"float blob has {len(buf)} bytes for {n_elem} elements")


def n_orbitals_def2svp(Z: np.ndarray) -> int:
    """5 functions for H/He (2s1p), 14 for C..F (3s2p1d), spherical."""
    return int(sum(5 if z <= 2 else 14 for z in Z))


def sample_ids(conn, n_sample: int, rng, stratify: bool = True):
    """Choose molecule ids.  With stratify=True the sample is spread as evenly
    as possible over the number-of-atoms N so that large molecules (the ones
    that stress lambda_min(S)) are not under-represented."""
    rows = conn.execute("SELECT id, N FROM data").fetchall()
    ids = np.array([r[0] for r in rows])
    Ns = np.array([r[1] for r in rows])
    if n_sample <= 0 or n_sample >= len(ids):
        return ids
    if not stratify:
        return rng.choice(ids, size=n_sample, replace=False)

    groups = {}
    for i, n in zip(ids, Ns):
        groups.setdefault(int(n), []).append(int(i))
    per_group = max(1, n_sample // len(groups))
    chosen = []
    for n in sorted(groups):
        g = groups[n]
        take = min(per_group, len(g))
        chosen += list(rng.choice(g, size=take, replace=False))
    # top up randomly from the remainder if stratification left us short
    if len(chosen) < n_sample:
        remaining = np.setdiff1d(ids, np.array(chosen))
        extra = rng.choice(remaining, size=n_sample - len(chosen), replace=False)
        chosen += list(extra)
    return np.array(sorted(chosen))


def fetch_molecule(conn, mol_id: int):
    row = conn.execute("SELECT N, Z, pos, Ham FROM data WHERE id=?",
                       (int(mol_id),)).fetchone()
    if row is None:
        raise KeyError(mol_id)
    N = int(row[0])
    Z = _decode_int(row[1], N)
    pos = _decode_float(row[2], 3 * N).reshape(N, 3)
    n_orb = n_orbitals_def2svp(Z)
    F = _decode_float(row[3], n_orb * n_orb).reshape(n_orb, n_orb)
    F = 0.5 * (F + F.T)
    return Z, pos, F


def build_mol(Z: np.ndarray, pos: np.ndarray) -> gto.Mole:
    atom = [[int(z), pos[i].tolist()] for i, z in enumerate(Z)]
    mol = gto.Mole()
    mol.build(verbose=0, atom=atom, basis=QH9_BASIS, unit="ang")
    return mol


# ==========================================================================
# Linear algebra (same definitions as conformalorb_phase0.py)
# ==========================================================================

def orthogonalizer(S, thresh=1e-7):
    s_eig, U = np.linalg.eigh(S)
    keep = s_eig > thresh
    X = U[:, keep] / np.sqrt(s_eig[keep])
    return X, s_eig, int((~keep).sum())


def solve_generalized(F, S, X):
    Ft = X.T @ F @ X
    Ft = 0.5 * (Ft + Ft.T)
    eps, Ct = np.linalg.eigh(Ft)
    return eps, X @ Ct


def frontier_window(eps, n_occ, n_below=2, n_above=2, degen_tol_ev=0.05):
    tol = degen_tol_ev / HARTREE_TO_EV
    i_lo = max(0, n_occ - n_below)
    i_hi = min(len(eps) - 1, n_occ + n_above - 1)
    while i_lo > 0 and (eps[i_lo] - eps[i_lo - 1]) < tol:
        i_lo -= 1
    while i_hi < len(eps) - 1 and (eps[i_hi + 1] - eps[i_hi]) < tol:
        i_hi += 1
    return i_lo, i_hi


def random_symmetric(n, spectral_norm, rng):
    G = rng.standard_normal((n, n))
    E = 0.5 * (G + G.T)
    return E * (spectral_norm / np.linalg.norm(E, 2))


# ==========================================================================
# Per-molecule analysis
# ==========================================================================

def analyse(Z, pos, F, rng, degen_tol_ev, ortho_thresh, probe_norm):
    mol = build_mol(Z, pos)
    if mol.nao_nr() != F.shape[0]:
        raise RuntimeError(f"AO count mismatch: PySCF {mol.nao_nr()} vs stored {F.shape[0]}")
    S = mol.intor_symmetric("int1e_ovlp")
    n_occ = mol.nelectron // 2

    X, s_eig, n_drop = orthogonalizer(S, ortho_thresh)
    eps, C = solve_generalized(F, S, X)

    i_lo, i_hi = frontier_window(eps, n_occ, degen_tol_ev=degen_tol_ev)
    gap_lo = eps[i_lo] - eps[i_lo - 1] if i_lo > 0 else np.inf
    gap_hi = eps[i_hi + 1] - eps[i_hi] if i_hi < len(eps) - 1 else np.inf
    delta0 = min(gap_lo, gap_hi)

    # typical (not worst-case) amplification of a Hamiltonian error by the
    # orthogonaliser, from ONE random probe direction
    dF = random_symmetric(F.shape[0], probe_norm, rng)
    amp_typical = np.linalg.norm(X.T @ dF @ X, 2) / probe_norm
    amp_worst = 1.0 / s_eig.min()

    n_heavy = int((Z > 1).sum())
    return dict(
        n_atoms=len(Z), n_heavy=n_heavy, n_basis=int(F.shape[0]), n_occ=n_occ,
        lambda_min_S=float(s_eig.min()), cond_S=float(s_eig.max() / s_eig.min()),
        n_dropped=n_drop,
        amp_typical=float(amp_typical), amp_worst=float(amp_worst),
        e_homo_eV=float(eps[n_occ - 1] * HARTREE_TO_EV),
        e_lumo_eV=float(eps[n_occ] * HARTREE_TO_EV),
        gap_eV=float((eps[n_occ] - eps[n_occ - 1]) * HARTREE_TO_EV),
        homo_split_eV=float((eps[n_occ - 1] - eps[n_occ - 2]) * HARTREE_TO_EV),
        lumo_split_eV=float((eps[n_occ + 1] - eps[n_occ]) * HARTREE_TO_EV),
        k_window=int(i_hi - i_lo + 1), i_lo=int(i_lo), i_hi=int(i_hi),
        delta0_Ha=float(delta0), delta0_eV=float(delta0 * HARTREE_TO_EV),
        n_negative_eigs=int((eps < 0).sum()),
    ), (mol, S, eps)


def fresh_scf_check(mol, eps_stored):
    """Re-run B3LYP/def2-SVP with TIGHT convergence and compare its mo_energy
    with the eigenvalues obtained from the STORED Fock matrix + recomputed S.

    How to read the number:
      ~1e-1 .. 1  Ha  : AO ordering / basis mismatch  -> bug, stop.
      ~1e-6 .. 1e-5 Ha: the stored matrix was generated with PySCF's default
                        conv_tol (1e-9); mf.get_fock() after kernel() is built
                        from the final density without DIIS, so its eigenvalues
                        differ from mo_energy at the convergence-noise level.
                        This is the IRREDUCIBLE floor of any certificate on eps.
      ~1e-9  Ha       : stored matrix generated with tight convergence.
      a near-uniform ~1e-3 Ha offset: the definition of 'b3lyp' (VWN3 vs VWN5
                        correlation) differs between PySCF versions; affects F,
                        never S; harmless for the survey.
    """
    mf = dft.RKS(mol)
    mf.xc = QH9_XC
    mf.verbose = 0
    mf.conv_tol = 1e-13
    mf.kernel()
    return float(np.max(np.abs(mf.mo_energy - eps_stored))), mf.converged


# ==========================================================================
# Reporting
# ==========================================================================

def q(a, p):
    return float(np.quantile(a, p))


def report(rows, degen_tols, ml_error_levels):
    lam = np.array([r["lambda_min_S"] for r in rows])
    cond = np.array([r["cond_S"] for r in rows])
    amp_t = np.array([r["amp_typical"] for r in rows])
    amp_w = np.array([r["amp_worst"] for r in rows])
    gap = np.array([r["gap_eV"] for r in rows])
    hs = np.array([r["homo_split_eV"] for r in rows])
    ls = np.array([r["lumo_split_eV"] for r in rows])
    k = np.array([r["k_window"] for r in rows])
    d0 = np.array([r["delta0_Ha"] for r in rows])
    nh = np.array([r["n_heavy"] for r in rows])
    n = len(rows)

    print("\n" + "=" * 78)
    print(f"Q1  OVERLAP CONDITIONING   (n = {n})")
    print("=" * 78)
    print(f"{'quantile':<12}{'lambda_min(S)':>16}{'cond(S)':>12}"
          f"{'amp typical':>13}{'amp worst':>12}")
    for p in (0.05, 0.25, 0.50, 0.75, 0.95, 1.00):
        print(f"{p:<12.2f}{q(lam, p):>16.3e}{q(cond, p):>12.0f}"
              f"{q(amp_t, p):>13.1f}{q(amp_w, p):>12.0f}")
    print("\nby number of heavy atoms:")
    print(f"{'n_heavy':>8}{'count':>8}{'median lam_min':>16}{'median cond':>13}{'median amp_typ':>16}")
    for h in sorted(set(nh)):
        m = nh == h
        print(f"{h:>8}{m.sum():>8}{np.median(lam[m]):>16.3e}"
              f"{np.median(cond[m]):>13.0f}{np.median(amp_t[m]):>16.1f}")
    print("\n-> 'amp worst' = 1/lambda_min(S) is the factor the LOOSE bound pays;")
    print("   'amp typical' is what a random Hamiltonian error actually suffers.")
    print("   If worst >> typical, conformalise ||X^T dF X||_2 directly, never ||dF||/lambda_min.")

    print("\n" + "=" * 78)
    print("Q2  FRONTIER DEGENERACY")
    print("=" * 78)
    print(f"{'tolerance (eV)':<16}{'HOMO pair':>12}{'LUMO pair':>12}{'either':>12}")
    for t in degen_tols:
        a = hs < t
        b = ls < t
        print(f"{t:<16.3f}{100*a.mean():>11.1f}%{100*b.mean():>11.1f}%{100*(a|b).mean():>11.1f}%")
    print("\nadaptive window size k (base = 4):")
    for kk in sorted(set(k)):
        print(f"   k = {kk:<3d} {100*(k==kk).mean():6.1f}%")
    print("-> every percent here is a molecule where the per-orbital overlap score is ill-defined.")

    print("\n" + "=" * 78)
    print("Q3  DAVIS-KAHAN ADMISSIBILITY   (certificate vacuous if ||dF_tilde||_2 > delta0)")
    print("=" * 78)
    print(f"delta0 quantiles (Ha):  "
          + "  ".join(f"p{int(100*p):02d}={q(d0,p):.2e}" for p in (0.05, 0.25, 0.5, 0.75, 0.95)))
    print(f"\n{'||dF_tilde||_2 (Ha)':<22}{'fraction VACUOUS':>18}")
    for e in ml_error_levels:
        print(f"{e:<22.0e}{100*(d0 < e).mean():>17.1f}%")
    print("-> the analytic route is silent exactly on these molecules; the empirical")
    print("   route (split CP on eps and on s_B) must carry them.")

    print("\n" + "=" * 78)
    print("Q4  HOMO-LUMO GAP  ->  Mondrian bins")
    print("=" * 78)
    print("gap quantiles (eV): "
          + "  ".join(f"p{int(100*p):02d}={q(gap,p):.2f}" for p in (0.05, 0.25, 0.5, 0.75, 0.95)))
    edges = [q(gap, p) for p in (0.25, 0.5, 0.75)]
    print(f"suggested quartile edges: {edges[0]:.2f} | {edges[1]:.2f} | {edges[2]:.2f} eV")

    sanity = [r for r in rows if r["n_negative_eigs"] < r["n_occ"] or r["e_homo_eV"] > 0]
    print(f"\nsanity: {len(sanity)} molecules with a non-negative HOMO or fewer negative"
          f" eigenvalues than occupied orbitals (expected 0).")


def make_plot(rows, path):
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib not available; skipping plot")
        return
    lam = np.array([r["lambda_min_S"] for r in rows])
    nh = np.array([r["n_heavy"] for r in rows])
    gap = np.array([r["gap_eV"] for r in rows])
    hs = np.array([r["homo_split_eV"] for r in rows])
    d0 = np.array([r["delta0_eV"] for r in rows])
    amp_t = np.array([r["amp_typical"] for r in rows])
    amp_w = np.array([r["amp_worst"] for r in rows])

    fig, ax = plt.subplots(2, 2, figsize=(10, 8))
    ax[0, 0].scatter(nh + 0.15 * np.random.randn(len(nh)), lam, s=3, alpha=0.3)
    ax[0, 0].set_yscale("log"); ax[0, 0].set_xlabel("heavy atoms"); ax[0, 0].set_ylabel(r"$\lambda_{\min}(S)$")
    ax[0, 0].set_title("Overlap conditioning vs size")
    ax[0, 1].scatter(amp_w, amp_t, s=3, alpha=0.3)
    ax[0, 1].set_xscale("log"); ax[0, 1].set_yscale("log")
    ax[0, 1].set_xlabel(r"worst-case amplification $1/\lambda_{\min}(S)$")
    ax[0, 1].set_ylabel("typical amplification (random probe)")
    ax[0, 1].set_title("Loose bound vs reality")
    ax[1, 0].hist(np.log10(np.maximum(hs, 1e-6)), bins=60)
    ax[1, 0].set_xlabel(r"$\log_{10}$ HOMO pair split (eV)"); ax[1, 0].set_ylabel("count")
    ax[1, 0].set_title("Frontier degeneracy")
    ax[1, 1].scatter(gap, d0, s=3, alpha=0.3)
    ax[1, 1].set_yscale("log"); ax[1, 1].set_xlabel("HOMO-LUMO gap (eV)")
    ax[1, 1].set_ylabel(r"$\delta_0$ around frontier block (eV)")
    ax[1, 1].set_title("Davis-Kahan admissible error")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    print(f"figure written: {path}")


# ==========================================================================
# Driver
# ==========================================================================

def main():
    p = argparse.ArgumentParser(description="ConformalOrb Phase 1a: QH9 survey")
    p.add_argument("--db", required=True, help="path to QH9Stable.db")
    p.add_argument("--n-sample", type=int, default=20000,
                   help="number of molecules to survey (<=0 = all)")
    p.add_argument("--no-stratify", action="store_true")
    p.add_argument("--degen-tol", type=float, default=0.05, help="eV")
    p.add_argument("--ortho-thresh", type=float, default=1e-7)
    p.add_argument("--probe-norm", type=float, default=1e-4, help="Ha")
    p.add_argument("--check-scf", type=int, default=3,
                   help="re-run SCF on this many molecules to verify ordering")
    p.add_argument("--seed", type=int, default=20260904)
    p.add_argument("--out", default="qh9_survey.csv")
    p.add_argument("--plot", action="store_true")
    args = p.parse_args()

    rng = np.random.default_rng(args.seed)
    conn = sqlite3.connect(args.db)
    ids = sample_ids(conn, args.n_sample, rng, stratify=not args.no_stratify)
    print(f"surveying {len(ids)} molecules from {args.db}")

    rows, t0 = [], time.time()
    checks_left = args.check_scf
    for j, mol_id in enumerate(ids):
        try:
            Z, pos, F = fetch_molecule(conn, mol_id)
            r, (mol, S, eps) = analyse(Z, pos, F, rng, args.degen_tol,
                                       args.ortho_thresh, args.probe_norm)
        except Exception as exc:                      # keep going, log it
            print(f"  [skip] id={mol_id}: {exc}")
            continue
        r = {"id": int(mol_id), **r}
        if checks_left > 0:
            d, ok = fresh_scf_check(mol, eps)
            r["scf_check_max_deps_Ha"] = d
            print(f"  [check] id={mol_id}: max|eps_fresh - eps_stored| = {d:.2e} Ha"
                  f" (converged={ok})")
            checks_left -= 1
        rows.append(r)
        if (j + 1) % 1000 == 0:
            rate = (j + 1) / (time.time() - t0)
            print(f"  {j+1}/{len(ids)}  ({rate:.1f} mol/s, "
                  f"~{(len(ids)-j-1)/rate/60:.1f} min left)")

    keys = sorted({k for r in rows for k in r}, key=lambda s: (s != "id", s))
    with open(args.out, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print(f"\n{len(rows)} molecules written to {args.out}  "
          f"({time.time()-t0:.0f} s)")

    report(rows, degen_tols=(0.01, 0.05, 0.10),
           ml_error_levels=(1e-4, 1e-3, 5e-3, 1e-2))
    if args.plot:
        make_plot(rows, os.path.splitext(args.out)[0] + ".png")


if __name__ == "__main__":
    main()


Writing co_survey.py


In [7]:
%%writefile co_infer.py
# === Phase 1b: QHNet inference on the ConformalOrb subset =================================
import os, sys, types, time, math, sqlite3
import numpy as np
import torch

HARTREE_TO_EV = 27.211386245988
QH9_XC = "b3lyp5"          # QH9-stable was generated with PySCF 2.2.1 'b3lyp' == VWN5 variant == 'b3lyp5' in PySCF >= 2.3

# ---------------------------------------------------------------- 1. dependency shims
def _radius_graph_torch(x, r, batch=None, loop=False, max_num_neighbors=32, flow="source_to_target", **kw):
    """Pure-torch replacement for torch_cluster.radius_graph with the SAME edge ordering:
    rows [neighbour j, centre i], ordered by i (major) then j (minor), i != j, within radius r."""
    if batch is None:
        batch = torch.zeros(x.shape[0], dtype=torch.long, device=x.device)
    n = x.shape[0]
    d2 = torch.cdist(x, x) ** 2
    same = batch[:, None] == batch[None, :]
    keep = same & (d2 <= r * r) & ~torch.eye(n, dtype=torch.bool, device=x.device)
    ii, jj = torch.nonzero(keep, as_tuple=True)          # nonzero returns row-major order: i major, j minor
    if max_num_neighbors is not None and int(max_num_neighbors) < n:  # cap like torch_cluster (first hits per centre)
        counts = torch.zeros(n, dtype=torch.long, device=x.device)
        order = torch.arange(len(ii), device=x.device)
        rank = order - torch.searchsorted(ii, ii, right=False)        # position within each centre's block
        m = rank < int(max_num_neighbors); ii, jj = ii[m], jj[m]
    return torch.stack([jj, ii], dim=0)

def install_shims():
    """Register torch_cluster / torch_scatter substitutes only if the real packages are missing.
    torch_geometric is imported first so that its optional-dependency detection sees the true state."""
    import importlib.machinery
    from torch_geometric.utils import scatter as _pyg_scatter
    def _shim(name):
        m = types.ModuleType(name); m.__spec__ = importlib.machinery.ModuleSpec(name, None); return m
    try:
        import torch_cluster  # noqa
        print("torch_cluster: native")
    except ImportError:
        m = _shim("torch_cluster"); m.radius_graph = _radius_graph_torch
        sys.modules["torch_cluster"] = m; print("torch_cluster: pure-torch shim")
    try:
        import torch_scatter  # noqa
        print("torch_scatter: native")
    except ImportError:
        m = _shim("torch_scatter")
        m.scatter = lambda src, index, dim=-1, out=None, dim_size=None, reduce="sum": _pyg_scatter(src, index, dim=dim, dim_size=dim_size, reduce=reduce)
        sys.modules["torch_scatter"] = m; print("torch_scatter: torch_geometric shim")

# ---------------------------------------------------------------- 2. AO-order conventions (numpy port of the official code)
_CONV = {
    "pyscf_def2svp": dict(atom_to_orbitals_map={1: "ssp", 6: "sssppd", 7: "sssppd", 8: "sssppd", 9: "sssppd"},
                          orbital_idx_map={"s": [0], "p": [1, 2, 0], "d": [0, 1, 2, 3, 4]}),
    "back2pyscf":    dict(atom_to_orbitals_map={1: "ssp", 6: "sssppd", 7: "sssppd", 8: "sssppd", 9: "sssppd"},
                          orbital_idx_map={"s": [0], "p": [2, 0, 1], "d": [0, 1, 2, 3, 4]}),
}

def transform_indices(atoms, convention):
    conv = _CONV[convention]; orbitals = "".join(conv["atom_to_orbitals_map"][int(a)] for a in atoms)
    idx, off = [], 0
    for orb in orbitals:
        m = conv["orbital_idx_map"][orb]; idx.append(np.array(m) + off); off += len(m)
    return np.concatenate(idx).astype(int)

def matrix_transform(M, atoms, convention):
    t = transform_indices(atoms, convention)
    return M[..., t, :][..., :, t]

# ---------------------------------------------------------------- 3. model
QHNET_KWARGS = dict(in_node_features=1, sh_lmax=4, hidden_size=128, bottle_hidden_size=32,
                    num_gnn_layers=5, max_radius=15, num_nodes=10, radius_embed_dim=16)   # exactly as in the official test.py

def get_model(ckpt_path=None, device="cpu"):
    from models import QHNet                       # models/ = the official package (ori_QHNet_with_bias)
    model = QHNet(**QHNET_KWARGS)
    if ckpt_path:
        state = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        state = state.get("state_dict", state)
        missing, unexpected = model.load_state_dict(state, strict=False)
        assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
        print("checkpoint loaded:", os.path.basename(ckpt_path))
    model.set(torch.device(device)); model.eval()
    return model

# ---------------------------------------------------------------- 4. data
def edge_index_full(n):
    """Official ordering: for i (src) in range(n): for j (dst) != i: (j, i)  ->  rows [dst, src]."""
    src = np.repeat(np.arange(n), n - 1)
    dst = np.concatenate([np.delete(np.arange(n), i) for i in range(n)])
    return torch.tensor(np.stack([dst, src]), dtype=torch.long)

def make_batch(Z, pos, dtype=torch.float32):
    from torch_geometric.data import Data, Batch
    d = Data(pos=torch.tensor(np.asarray(pos), dtype=dtype),
             atoms=torch.tensor(np.asarray(Z), dtype=torch.long).view(-1, 1),
             edge_index_full=edge_index_full(len(Z)))
    return Batch.from_data_list([d])

def fast_build_final_matrix(model, batch, diag, nondiag):
    """Vectorised equivalent of model.build_final_matrix for a single-graph batch.
    Edge k of edge_index_full is (dst=j, src=i) at position i*(n-1) + j - (j>i)."""
    atoms = batch.atoms.view(-1).tolist(); n = len(atoms)
    masks = [model.orbital_mask[a] for a in atoms]; sizes = [len(m) for m in masks]
    offs = np.concatenate([[0], np.cumsum(sizes)]); N = int(offs[-1])
    H = torch.zeros(N, N, dtype=diag.dtype, device=diag.device)
    for i in range(n):
        for j in range(n):
            if i == j:
                blk = diag[i]
            else:
                k = i * (n - 1) + j - (1 if j > i else 0)
                blk = nondiag[k]
            # official: matrix_col.append(block.index_select(-2, mask[dst=j]).index_select(-1, mask[src=i])); rows=dst, cols=src
            H[offs[j]:offs[j + 1], offs[i]:offs[i + 1]] = blk.index_select(-2, masks[j]).index_select(-1, masks[i])
    return H.unsqueeze(0)

@torch.no_grad()
def predict_H(model, Z, pos, device, use_fast=True, check_against_official=False):
    """Returns Hhat in PySCF AO order (float64 numpy), plus wall time of the forward pass."""
    batch = make_batch(Z, pos).to(device)
    t0 = time.time(); out = model(batch); t_fwd = time.time() - t0
    diag, nondiag = out["hamiltonian_diagonal_blocks"], out["hamiltonian_non_diagonal_blocks"]
    H = fast_build_final_matrix(model, batch, diag, nondiag) if use_fast else model.build_final_matrix(batch, diag, nondiag)
    if check_against_official:
        H_off = model.build_final_matrix(batch, diag, nondiag)
        assert torch.allclose(H, H_off, atol=1e-6, rtol=0), "fast assembly differs from the official build_final_matrix"
    H = H[0].double().cpu().numpy()
    H = matrix_transform(H, np.asarray(Z), "back2pyscf")
    return 0.5 * (H + H.T), t_fwd

# ---------------------------------------------------------------- 5. metrics (same definitions as Phase 0)
def orthogonalizer(S, thresh=1e-7):
    s, U = np.linalg.eigh(S); keep = s > thresh
    return U[:, keep] / np.sqrt(s[keep]), s

def solve_generalized(F, S, X):
    Ft = X.T @ F @ X; Ft = 0.5 * (Ft + Ft.T); e, Ct = np.linalg.eigh(Ft); return e, X @ Ct

def expand_window(eps, i_lo, i_hi, tol):
    while i_lo > 0 and eps[i_lo] - eps[i_lo - 1] < tol: i_lo -= 1
    while i_hi < len(eps) - 1 and eps[i_hi + 1] - eps[i_hi] < tol: i_hi += 1
    return i_lo, i_hi

def principal_cosines(C_ref, C_pert, S, sl):
    M = C_ref[:, sl].T @ S @ C_pert[:, sl]; return np.clip(np.linalg.svd(M, compute_uv=False), 0, 1)

def subspace_block(eps, C, C_hat, S, i_lo, i_hi, E_norm):
    """Score s_B, sin(theta_max), Davis-Kahan gap and bound for the block [i_lo, i_hi]."""
    sl = slice(i_lo, i_hi + 1); sig = principal_cosines(C, C_hat, S, sl); k = i_hi - i_lo + 1
    s_B = math.sqrt(max(0.0, 1 - float(np.sum(sig ** 2)) / k)); sin_max = math.sqrt(max(0.0, 1 - float(sig.min()) ** 2))
    gap_lo = eps[i_lo] - eps[i_lo - 1] if i_lo > 0 else np.inf
    gap_hi = eps[i_hi + 1] - eps[i_hi] if i_hi < len(eps) - 1 else np.inf
    delta = min(gap_lo, gap_hi); eff = delta - E_norm
    bound = E_norm / eff if eff > 0 else np.inf
    return dict(k=k, s_B=s_B, sin_theta=sin_max, delta=float(delta), dk_bound=float(bound),
                dk_vacuous=not np.isfinite(bound), dk_holds=bool(sin_max <= bound + 1e-12) if np.isfinite(bound) else None)

def frontier_metrics(H, H_hat, S, n_occ, degen_tol_ev=0.05):
    tol = degen_tol_ev / HARTREE_TO_EV
    X, s_eig = orthogonalizer(S)
    eps, C = solve_generalized(H, S, X); eps_h, C_h = solve_generalized(H_hat, S, X)
    dH = H_hat - H; E = X.T @ dH @ X; E = 0.5 * (E + E.T)
    E_norm = float(np.linalg.norm(E, 2))
    r = dict(n_basis=H.shape[0], n_occ=n_occ, lambda_min_S=float(s_eig.min()),
             H_mae=float(np.abs(dH).mean()), H_max=float(np.abs(dH).max()), dH_spec=float(np.linalg.norm(dH, 2)),
             E_spec=E_norm, amp_real=E_norm / max(float(np.linalg.norm(dH, 2)), 1e-300),
             eps_mae_occ=float(np.abs(eps_h[:n_occ] - eps[:n_occ]).mean()), eps_max_all=float(np.abs(eps_h - eps).max()),
             weyl_holds=bool(np.abs(eps_h - eps).max() <= E_norm + 1e-12),
             e_homo=float(eps[n_occ - 1]), e_lumo=float(eps[n_occ]),
             d_homo=float(eps_h[n_occ - 1] - eps[n_occ - 1]), d_lumo=float(eps_h[n_occ] - eps[n_occ]),
             d_gap=float((eps_h[n_occ] - eps_h[n_occ - 1]) - (eps[n_occ] - eps[n_occ - 1])),
             homo_split=float(eps[n_occ - 1] - eps[n_occ - 2]), lumo_split=float(eps[n_occ + 1] - eps[n_occ]),
             cos_homo=float(abs(C[:, n_occ - 1] @ S @ C_h[:, n_occ - 1])),      # naive per-orbital score, for comparison
             cos_lumo=float(abs(C[:, n_occ] @ S @ C_h[:, n_occ])),
             # the official QH9 "psi" metric: Euclidean cosine of AO coefficient vectors, |.| averaged over occupied orbitals
             psi_cos_occ=float(np.mean(np.abs(np.sum(C[:, :n_occ] * C_h[:, :n_occ], axis=0)
                                              / (np.linalg.norm(C[:, :n_occ], axis=0) * np.linalg.norm(C_h[:, :n_occ], axis=0))))))
    blocks = {"homo": (n_occ - 1, n_occ - 1), "lumo": (n_occ, n_occ), "pair": (n_occ - 1, n_occ), "win4": (n_occ - 2, n_occ + 1)}
    for name, (lo, hi) in blocks.items():
        lo, hi = expand_window(eps, lo, hi, tol)
        for key, val in subspace_block(eps, C, C_h, S, lo, hi, E_norm).items():
            r[f"{name}_{key}"] = val
    r.update(mo_basis_diagnostics(dH, C, eps, n_occ))
    # test-time-available quantities (from the PREDICTED spectrum) for gap-normalised conformal scores
    r.update(gap_hat=float(eps_h[n_occ] - eps_h[n_occ - 1]),
             homo_split_hat=float(eps_h[n_occ - 1] - eps_h[n_occ - 2]), lumo_split_hat=float(eps_h[n_occ + 1] - eps_h[n_occ]))
    return r


def mo_basis_diagnostics(dH, C, eps, n_occ, f_lo=None, f_hi=None):
    """Where does the error live?  D = C^T dH C is the Hamiltonian error expressed in the reference MO basis
    (C^T S C = I, so ||D||_2 = ||X^T dH X||_2).  Block operator norms tell whether the error sits in the
    chemically relevant frontier/occupied space or in the high-energy virtual space; first- and second-order
    perturbation theory (PT) estimates of the frontier shifts and rotations are recorded for comparison with
    the exact values and with the global Weyl / Davis-Kahan bounds."""
    D = C.T @ dH @ C; D = 0.5 * (D + D.T); n = D.shape[0]
    f_lo = n_occ - 2 if f_lo is None else f_lo; f_hi = n_occ + 1 if f_hi is None else f_hi
    nrm2 = lambda M: float(np.linalg.norm(M, 2)) if M.size else 0.0
    fro2 = float(np.sum(D ** 2))
    imax = np.unravel_index(int(np.argmax(np.abs(D))), D.shape)
    out = dict(D_ff=nrm2(D[f_lo:f_hi + 1, f_lo:f_hi + 1]), D_oo=nrm2(D[:n_occ, :n_occ]), D_vv=nrm2(D[n_occ:, n_occ:]),
               D_ov=nrm2(D[:n_occ, n_occ:]), D_vv_frac=float(np.sum(D[n_occ:, n_occ:] ** 2) / fro2) if fro2 > 0 else np.nan,
               D_max_eps_i=float(eps[imax[0]]), D_max_eps_j=float(eps[imax[1]]))
    for name, i in (("homo", n_occ - 1), ("lumo", n_occ)):
        den = eps[i] - eps; den[i] = np.inf
        pt1 = float(D[i, i]); pt2 = pt1 + float(np.sum(D[i, :] ** 2 / den))
        mix = D[i, :] / den; mix[i] = 0.0; m = float(np.linalg.norm(mix))
        gap_loc = min(eps[i] - eps[i - 1], eps[i + 1] - eps[i]); coup = float(np.linalg.norm(np.delete(D[i, :], i)))
        out.update({f"{name}_pt1": pt1, f"{name}_pt2": pt2, f"{name}_sin_pt": m / math.sqrt(1 + m * m),
                    f"{name}_coupling": coup, f"{name}_gap_local": float(gap_loc),
                    f"{name}_dk_local": coup / (gap_loc - abs(pt1)) if gap_loc > abs(pt1) else np.inf})
    return out

# ---------------------------------------------------------------- 6. subset reader
def n_orbitals_def2svp(Z): return int(sum(5 if z <= 2 else 14 for z in Z))

def unpack_upper(packed, n):
    F = np.zeros((n, n)); F[np.triu_indices(n)] = packed; return F + F.T - np.diag(np.diag(F))

def read_subset(conn, mol_id):
    N, Zb, posb, Hb = conn.execute("SELECT N, Z, pos, Ham FROM data WHERE id=?", (int(mol_id),)).fetchone()
    Z = np.frombuffer(Zb, np.int32 if len(Zb) == 4 * N else np.int64).astype(int)
    pos = np.frombuffer(posb, np.float64).reshape(N, 3); n = n_orbitals_def2svp(Z)
    H = unpack_upper(np.frombuffer(Hb, np.float64), n) if len(Hb) == 8 * n * (n + 1) // 2 else np.frombuffer(Hb, np.float64).reshape(n, n)
    return Z, pos, 0.5 * (H + H.T)

def overlap_and_nocc(Z, pos):
    from pyscf import gto
    mol = gto.Mole(); mol.build(verbose=0, atom=[[int(z), pos[i].tolist()] for i, z in enumerate(Z)], basis="def2svp", unit="ang")
    return mol.intor_symmetric("int1e_ovlp"), mol.nelectron // 2


Writing co_infer.py


In [8]:
%%writefile co_fock.py
# === Phase 2 / Part A: one Fock build from the predicted density ===============================
# For every (molecule, checkpoint) with a stored prediction Hhat:
#   P_hat = 2 C_occ C_occ^T from the predicted occupied orbitals  ->  F1 = Fock(P_hat)  (b3lyp5/def2-SVP)
# Features (all available at test time, no reference needed):
#   f1_res_fro / f1_res_max      ||F1 - Hhat||_F, max|F1 - Hhat|           self-consistency residual
#   f1_res_occ / f1_res_frontier  block operator norms of C_hat^T (F1 - Hhat) C_hat (occupied / frontier window)
#   f1_grad                       ||F1 P S - S P F1||_F                       SCF gradient at P_hat
#   f1_pt1_homo / f1_pt1_lumo     c_i^T (F1 - Hhat) c_i                       first-order PT estimate of the shift
#   f1_dhomo / f1_dlumo / f1_gap  eps(F1) - eps(Hhat) for HOMO / LUMO, and the F1 gap
#   f1_sB_homo / f1_sB_lumo       subspace distance between Hhat's and F1's HOMO / LUMO
# Optionally (ref=True, for validation only) the same quantities of F1 against the reference H.
import os, sys, time, math, sqlite3
import numpy as np

QH9_XC = "b3lyp5"

def _n_orb(Z): return int(sum(5 if z <= 2 else 14 for z in Z))

def _unpack(packed, n):
    F = np.zeros((n, n)); F[np.triu_indices(n)] = packed; return F + F.T - np.diag(np.diag(F))

def _ortho(S, thresh=1e-7):
    s, U = np.linalg.eigh(S); keep = s > thresh; return U[:, keep] / np.sqrt(s[keep])

def _solve(F, S, X):
    Ft = X.T @ F @ X; Ft = 0.5 * (Ft + Ft.T); e, Ct = np.linalg.eigh(Ft); return e, X @ Ct

def _window(e, i, tol=0.05 / 27.211386245988):
    lo = hi = i
    while lo > 0 and e[lo] - e[lo - 1] < tol: lo -= 1
    while hi < len(e) - 1 and e[hi + 1] - e[hi] < tol: hi += 1
    return lo, hi

def _sB_block(C_a, e_a, C_b, e_b, i, S):
    """Degeneracy-aware subspace distance between orbital i of spectrum a and of spectrum b (window = union)."""
    lo_a, hi_a = _window(e_a, i); lo_b, hi_b = _window(e_b, i); lo, hi = min(lo_a, lo_b), max(hi_a, hi_b)
    M = C_a[:, lo:hi + 1].T @ S @ C_b[:, lo:hi + 1]; sig = np.clip(np.linalg.svd(M, compute_uv=False), 0, 1)
    return math.sqrt(max(0.0, 1.0 - float(np.sum(sig ** 2)) / (hi - lo + 1)))

def fock_features(Z, pos, Hhat, grid_level=1, H_ref=None, use_df=False):
    """use_df=True builds J/K with density fitting (def2-universal-jkfit): ~2.5x faster, ||F_DF - F_exact||_F ~ 1e-3 Ha,
    i.e. 1-5% of a clean molecule's residual -- fine for the detector and the certificates, but keep one setting per checkpoint."""
    from pyscf import gto, dft, lib
    mol = gto.Mole(); mol.build(verbose=0, atom=[[int(z), pos[i].tolist()] for i, z in enumerate(Z)], basis="def2svp", unit="ang")
    S = mol.intor_symmetric("int1e_ovlp"); n_occ = mol.nelectron // 2; X = _ortho(S)
    eh, Ch = _solve(Hhat, S, X)
    P = 2.0 * Ch[:, :n_occ] @ Ch[:, :n_occ].T
    mf = dft.RKS(mol); mf.xc = QH9_XC; mf.grids.level = grid_level
    if use_df:
        mf = mf.density_fit()
    t0 = time.time(); F1 = mf.get_fock(dm=P); t_f = time.time() - t0; F1 = 0.5 * (F1 + F1.T)
    R = F1 - Hhat; e1, C1 = _solve(F1, S, X)
    D = Ch.T @ R @ Ch; f = slice(n_occ - 2, n_occ + 2)
    out = dict(n_basis=int(S.shape[0]), n_occ=n_occ, t_fock_s=t_f, method=f"{'df' if use_df else 'exact'}-g{grid_level}",
               f1_res_fro=float(np.linalg.norm(R)), f1_res_max=float(np.abs(R).max()),
               f1_res_occ=float(np.linalg.norm(D[:n_occ, :n_occ], 2)), f1_res_frontier=float(np.linalg.norm(D[f, f], 2)),
               f1_grad=float(np.linalg.norm(F1 @ P @ S - S @ P @ F1)),
               f1_pt1_homo=float(D[n_occ - 1, n_occ - 1]), f1_pt1_lumo=float(D[n_occ, n_occ]),
               f1_dhomo=float(e1[n_occ - 1] - eh[n_occ - 1]), f1_dlumo=float(e1[n_occ] - eh[n_occ]),
               f1_gap=float(e1[n_occ] - e1[n_occ - 1]), gap_hat=float(eh[n_occ] - eh[n_occ - 1]),
               f1_sB_homo=_sB_block(Ch, eh, C1, e1, n_occ - 1, S), f1_sB_lumo=_sB_block(Ch, eh, C1, e1, n_occ, S))
    if H_ref is not None:                      # validation only: is F1 closer to the truth than Hhat?
        e, C = _solve(H_ref, S, X)
        out.update(ref_dhomo_F1=float(e1[n_occ - 1] - e[n_occ - 1]), ref_dlumo_F1=float(e1[n_occ] - e[n_occ]),
                   ref_res_true=float(np.linalg.norm(H_ref - Hhat)))
    return out

# ---------------------------------------------------------------- worker
_W = {}
def _init_worker(subset_path, pred_path, grid_level, with_ref, use_df=False):
    os.environ["OMP_NUM_THREADS"] = "1"
    from pyscf import lib; lib.num_threads(1)
    try:                                              # BLAS threads are set at library load; limit them at run time
        from threadpoolctl import threadpool_limits; threadpool_limits(1)
    except Exception:
        pass
    _W["sub"] = sqlite3.connect(f"file:{subset_path}?mode=ro", uri=True)
    _W["pred"] = sqlite3.connect(f"file:{pred_path}?mode=ro", uri=True)
    _W["grid"] = grid_level; _W["ref"] = with_ref; _W["df"] = use_df

def _job(item):
    mol_id, ckpt = item
    try:
        N, Zb, posb, Hb = _W["sub"].execute("SELECT N, Z, pos, Ham FROM data WHERE id=?", (int(mol_id),)).fetchone()
        Z = np.frombuffer(Zb, np.int32 if len(Zb) == 4 * N else np.int64).astype(int); pos = np.frombuffer(posb, np.float64).reshape(N, 3)
        n = _n_orb(Z)
        Hp = _W["pred"].execute("SELECT Ham FROM pred WHERE id=? AND ckpt=?", (int(mol_id), ckpt)).fetchone()[0]
        Hhat = _unpack(np.frombuffer(Hp, np.float32).astype(np.float64), n)
        H_ref = None
        if _W["ref"]:
            H_ref = _unpack(np.frombuffer(Hb, np.float64), n) if len(Hb) == 8 * n * (n + 1) // 2 else np.frombuffer(Hb, np.float64).reshape(n, n)
        row = fock_features(Z, pos, Hhat, _W["grid"], H_ref, _W["df"])
        return {"id": int(mol_id), "ckpt": ckpt, "n_atoms": int(N), **row}
    except Exception as exc:
        return {"id": int(mol_id), "ckpt": ckpt, "error": f"{type(exc).__name__}: {exc}"}

def run_part_a(subset_path, pred_path, items, out_csv, n_workers=4, grid_level=1, with_ref=False, chunk=25, use_df=False):
    """items: list of (id, ckpt). Resumable: rows already in out_csv are skipped. Writes incrementally."""
    import csv, multiprocessing as mp
    done = set()
    if os.path.exists(out_csv) and os.path.getsize(out_csv) > 0:
        with open(out_csv) as fh:
            done = {(int(r["id"]), r["ckpt"]) for r in csv.DictReader(fh)}
    todo = [it for it in items if (int(it[0]), it[1]) not in done]
    print(f"{len(items)} items, {len(done)} done, {len(todo)} to run with {n_workers} workers ({'DF' if use_df else 'exact'} J/K, grid level {grid_level})", flush=True)
    if not todo:
        return
    fh = open(out_csv, "a", newline=""); writer = None; t0 = time.time(); n_ok = n_err = 0
    if done:
        with open(out_csv) as f0: fields = next(csv.reader(f0))
        writer = csv.DictWriter(fh, fieldnames=fields, extrasaction="ignore")
    ctx = mp.get_context("fork")
    with ctx.Pool(n_workers, initializer=_init_worker, initargs=(subset_path, pred_path, grid_level, with_ref, use_df)) as pool:
        for k, row in enumerate(pool.imap_unordered(_job, todo, chunksize=chunk)):
            if "error" in row:
                n_err += 1; print("  [skip]", row["id"], row["ckpt"], row["error"][:120]); continue
            if writer is None:
                writer = csv.DictWriter(fh, fieldnames=list(row.keys())); writer.writeheader()
            writer.writerow(row); n_ok += 1
            if (k + 1) % 200 == 0:
                fh.flush(); rate = (k + 1) / (time.time() - t0)
                print(f"  {k+1}/{len(todo)}  ({rate:.2f} mol/s, ~{(len(todo)-k-1)/rate/60:.0f} min left)", flush=True)
    fh.close(); print(f"done: {n_ok} rows, {n_err} errors, {(time.time()-t0)/60:.1f} min -> {out_csv}")


Writing co_fock.py


In [9]:
%%writefile co_conformal.py
# === Phase 2: conformal certificates for learned frontier orbitals ==========================
import math
import numpy as np
import pandas as pd

HARTREE_TO_EV = 27.211386245988

# ---------------------------------------------------------------- split conformal
def conformal_quantile(cal_scores, alpha):
    """Finite-sample-valid (1-alpha) quantile of calibration nonconformity scores."""
    s = np.sort(np.asarray(cal_scores, dtype=float)); n = len(s)
    k = int(math.ceil((n + 1) * (1 - alpha)))
    return s[min(k, n) - 1] if k <= n else np.inf

def coverage(eval_scores, qhat):
    return float(np.mean(np.asarray(eval_scores) <= qhat))

def mondrian_quantiles(cal_scores, cal_groups, alpha):
    return {g: conformal_quantile(cal_scores[cal_groups == g], alpha) for g in np.unique(cal_groups)}

def mondrian_eval(eval_scores, eval_groups, qdict, fallback):
    q = np.array([qdict.get(g, fallback) for g in eval_groups]); return q

# ---------------------------------------------------------------- FDR-controlled selection
def conformal_pvalues_against_nulls(s_test, s_cal_null, n_cal=None):
    """Conformal selection p-values (Jin & Candes 2023, cfBH with the CLIPPED score).
    H0_i: molecule i has a gross failure ('null' = not worth selecting).  Calibration molecules WITH a gross failure
    carry their suspicion score s; those without carry +inf and can never be counted.  With the imputed test score
    s_i the p-value is
        p_i = (1 + #{j in cal_null : s_j <= s_i}) / (n_cal + 1),
    which satisfies P(p_i <= t, H0_i) <= t, and BH applied to these p-values controls the FDR of the selected set
    (Jin & Candes 2023, Thm 2.3/2.6).  Dividing by n_cal + 1 (all calibration molecules) instead of n_null + 1
    (the 'BH_sub' variant of Bates et al. 2023) is what gives useful power when gross failures are rare."""
    s_null = np.sort(np.asarray(s_cal_null, dtype=float)); n = len(s_null) if n_cal is None else int(n_cal)
    ranks = np.searchsorted(s_null, np.asarray(s_test, dtype=float), side="right")
    return (1.0 + ranks) / (n + 1.0)

def benjamini_hochberg(p, q):
    p = np.asarray(p, dtype=float); m = len(p); order = np.argsort(p); ps = p[order]
    thresh = q * np.arange(1, m + 1) / m; ok = np.where(ps <= thresh)[0]
    if len(ok) == 0:
        return np.zeros(m, dtype=bool)
    k = ok.max(); sel = np.zeros(m, dtype=bool); sel[order[:k + 1]] = True; return sel

# ---------------------------------------------------------------- targets
def build_targets(df, fock=False):
    """Nonconformity scores (Ha or dimensionless), their test-time normalisers, and truth labels."""
    t = {}
    gh = np.abs(df.gap_hat.values)
    homo_loc = np.minimum(np.abs(df.homo_split_hat.values), gh); lumo_loc = np.minimum(np.abs(df.lumo_split_hat.values), gh)
    t["eps_HOMO"] = dict(score=np.abs(df.d_homo.values), norm=None, unit=1e3, label="|Δε_HOMO| (mHa)")
    t["eps_LUMO"] = dict(score=np.abs(df.d_lumo.values), norm=None, unit=1e3, label="|Δε_LUMO| (mHa)")
    t["gap"] = dict(score=np.abs(df.d_gap.values), norm=None, unit=1e3, label="|Δgap| (mHa)")
    t["sB_HOMO"] = dict(score=df.homo_s_B.values, norm=1.0 / np.maximum(homo_loc, 1e-4), unit=1, label="s_B HOMO")
    t["sB_LUMO"] = dict(score=df.lumo_s_B.values, norm=1.0 / np.maximum(lumo_loc, 1e-4), unit=1, label="s_B LUMO")
    if fock:   # one-Fock-build, PT-informed normalisers: |first-order estimate of the shift| + floor
        t["eps_HOMO"]["norm"] = np.abs(df.f1_pt1_homo.values) + 2e-4
        t["eps_LUMO"]["norm"] = np.abs(df.f1_pt1_lumo.values) + 2e-4
        t["gap"]["norm"] = np.abs(df.f1_pt1_lumo.values - df.f1_pt1_homo.values) + 3e-4
        t["sB_HOMO"]["norm"] = (df.f1_res_fro.values + 1e-3) / np.maximum(homo_loc, 1e-4)
        t["sB_LUMO"]["norm"] = (df.f1_res_fro.values + 1e-3) / np.maximum(lumo_loc, 1e-4)
        if "ref_dhomo_F1" in df.columns:   # the ONE-STEP-CORRECTED surrogate F1 = F[P_hat]: certify its frontier too
            t["F1_HOMO"] = dict(score=np.abs(df.ref_dhomo_F1.values), norm=df.f1_res_fro.values + 1e-3, unit=1e3, label="|Δε_HOMO(F1)| (mHa)")
            t["F1_LUMO"] = dict(score=np.abs(df.ref_dlumo_F1.values), norm=df.f1_res_fro.values + 1e-3, unit=1e3, label="|Δε_LUMO(F1)| (mHa)")
            t["F1_gap"] = dict(score=np.abs(df.ref_dlumo_F1.values - df.ref_dhomo_F1.values), norm=df.f1_res_fro.values + 1e-3, unit=1e3, label="|Δgap(F1)| (mHa)")
    return t

def trust_score(df, fock=False, tau=None):
    """Test-time 'suspicion' score (larger = worse), aligned with the trust criterion: the largest ratio of a
    test-time error ESTIMATE to its threshold tau.  With one Fock build the estimates are the first-order PT
    shifts c_i^T (F1 - Hhat) c_i for the energies and residual / local-gap for the orbitals; without it, only the
    spectral proxy (a small predicted gap is suspicious) is available."""
    if not fock:
        return -df.gap_hat.values * HARTREE_TO_EV
    tau = tau or {"eps_HOMO": 2e-3, "eps_LUMO": 5e-3, "gap": 5e-3, "sB_HOMO": 0.1, "sB_LUMO": 0.2}
    gh = np.abs(df.gap_hat.values)
    homo_loc = np.maximum(np.minimum(np.abs(df.homo_split_hat.values), gh), 1e-4); lumo_loc = np.maximum(np.minimum(np.abs(df.lumo_split_hat.values), gh), 1e-4)
    est = np.stack([np.abs(df.f1_pt1_homo.values) / tau["eps_HOMO"], np.abs(df.f1_pt1_lumo.values) / tau["eps_LUMO"],
                    np.abs(df.f1_pt1_lumo.values - df.f1_pt1_homo.values) / tau["gap"],
                    (df.f1_res_fro.values / homo_loc) / tau["sB_HOMO"], (df.f1_res_fro.values / lumo_loc) / tau["sB_LUMO"],
                    df.f1_res_fro.values / 0.05,                   # gross density failure
                    df.f1_sB_lumo.values / 0.3, df.f1_sB_homo.values / 0.3,   # F1 disagrees with Hhat on a frontier orbital
                    np.abs(df.f1_dlumo.values) / tau["eps_LUMO"], np.abs(df.f1_dhomo.values) / tau["eps_HOMO"]], axis=1)
    return np.log(np.max(est, axis=1) + 1e-9)

FLAG_COLS = ("f1_sB_lumo", "f1_sB_homo", "f1_res_fro")     # F1-vs-Hhat frontier comparison + global self-consistency residual

def flag_thresholds(df_cal, clean_cal, pq=0.98):
    """Per-feature thresholds = pq-quantile over CLEAN calibration molecules (clean = no oracle intruder)."""
    return {c: float(np.quantile(df_cal[c].values[clean_cal], pq)) for c in FLAG_COLS}

def intruder_flag_testtime(df, fock=False, thresholds=None, gap_thresh_ev=4.0):
    if fock and thresholds is not None:
        return np.any([df[c].values > thresholds[c] for c in FLAG_COLS], axis=0)
    return df.gap_hat.values * HARTREE_TO_EV < gap_thresh_ev

# ---------------------------------------------------------------- experiment
def run_conformal(df, alphas=(0.1, 0.05), R=20, seed=0, fock=False, tau=None, tau_gross=None, fdr_levels=(0.002, 0.005, 0.01, 0.02, 0.05), q_route=0.005, verbose=True):
    """Repeated random calibration/evaluation halves.  Returns a results dict; prints a report.
    Two trust levels:  tau       = fine thresholds -> reported through the calibrated intervals;
                       tau_gross = gross-failure thresholds (intruders, eV-scale errors) -> the null hypothesis of the
                                   FDR-controlled declaration 'no gross failure', which is what route 1 requires."""
    rng = np.random.default_rng(seed); n = len(df); T = build_targets(df, fock)
    tau = tau or {"eps_HOMO": 2e-3, "eps_LUMO": 5e-3, "gap": 5e-3, "sB_HOMO": 0.1, "sB_LUMO": 0.2}
    tau_gross = tau_gross or {"eps_HOMO": 1e-2, "eps_LUMO": 2e-2, "gap": 2e-2, "sB_HOMO": 0.7, "sB_LUMO": 0.7}
    oracle_intr = (df.lumo_s_B.values > 0.7) | (df.homo_s_B.values > 0.7)
    good_fine = np.all([T[k]["score"] <= tau[k] for k in tau], axis=0)
    good = np.all([T[k]["score"] <= tau_gross[k] for k in tau_gross], axis=0)    # 'no gross failure' ground truth (Hhat)
    good_F1 = (np.all([T[k]["score"] <= tau_gross[k.replace("F1_", "eps_") if k != "F1_gap" else "gap"] for k in ("F1_HOMO", "F1_LUMO", "F1_gap")], axis=0)
               if "F1_HOMO" in T else None)
    gap_ev = df.gap_hat.values * HARTREE_TO_EV; s_trust = trust_score(df, fock, tau)
    res = {k: {a: {"marg": [], "norm": [], "mond_flag": [], "mond_gap": [],
                   "w_marg": [], "w_norm": [], "w_mond_flag": [], "w_mond_gap": [],
                   "cov_intr_marg": [], "cov_intr_mond": [], "cov_clean_marg": []} for a in alphas} for k in T}
    fdr = {q: {"selected": [], "fdp": [], "power": []} for q in fdr_levels}
    fdr_F1 = {q: {"selected": [], "fdp": [], "power": []} for q in fdr_levels} if good_F1 is not None else None
    routing = {"r1": [], "r2": [], "r3": [], "err_gap_r1_q95": [], "intr_in_r1": [], "fine_in_r1": [],
               "F1_r1": [], "F1_gap_q95": [], "F1_lumo_q95": []}
    for r in range(R):
        perm = rng.permutation(n); cal, ev = perm[: n // 2], perm[n // 2:]
        # test-time flags / groups
        if fock:
            flag = intruder_flag_testtime(df, True, flag_thresholds(df.iloc[cal], ~oracle_intr[cal], 0.98))
        else:
            flag = intruder_flag_testtime(df, False)
        edges = np.quantile(gap_ev[cal], [0.25, 0.5, 0.75]); gapq = np.digitize(gap_ev, edges)
        for k, tg in T.items():
            s = tg["score"]; nrm = tg["norm"]
            for a in alphas:
                q = conformal_quantile(s[cal], a); R_ = res[k][a]
                R_["marg"].append(coverage(s[ev], q)); R_["w_marg"].append(np.median(np.full(len(ev), q)))
                R_["cov_intr_marg"].append(coverage(s[ev][oracle_intr[ev]], q) if oracle_intr[ev].any() else np.nan)
                R_["cov_clean_marg"].append(coverage(s[ev][~oracle_intr[ev]], q))
                if nrm is not None:
                    qn = conformal_quantile((s / nrm)[cal], a); qi = qn * nrm[ev]
                    R_["norm"].append(float(np.mean(s[ev] <= qi))); R_["w_norm"].append(float(np.median(qi)))
                else:
                    R_["norm"].append(np.nan); R_["w_norm"].append(np.nan)
                qd = mondrian_quantiles(s[cal], flag[cal].astype(int), a); qi = mondrian_eval(None, flag[ev].astype(int), qd, q)
                R_["mond_flag"].append(float(np.mean(s[ev] <= qi))); R_["w_mond_flag"].append(float(np.median(qi[~flag[ev]])))
                R_["cov_intr_mond"].append(float(np.mean(s[ev][oracle_intr[ev]] <= qi[oracle_intr[ev]])) if oracle_intr[ev].any() else np.nan)
                qg = mondrian_quantiles(s[cal], gapq[cal], a); qi = mondrian_eval(None, gapq[ev], qg, q)
                R_["mond_gap"].append(float(np.mean(s[ev] <= qi))); R_["w_mond_gap"].append(float(np.median(qi)))
        # FDR-controlled trust declaration on the evaluation half
        nulls = s_trust[cal][~good[cal]]
        p = conformal_pvalues_against_nulls(s_trust[ev], nulls, n_cal=len(cal))
        for qlev in fdr_levels:
            sel = benjamini_hochberg(p, qlev); ns = sel.sum()
            fdr[qlev]["selected"].append(ns / len(ev)); fdr[qlev]["fdp"].append(float((~good[ev][sel]).mean()) if ns else 0.0)
            fdr[qlev]["power"].append(float(sel[good[ev]].mean()))
        if fdr_F1 is not None:
            p1 = conformal_pvalues_against_nulls(s_trust[ev], s_trust[cal][~good_F1[cal]], n_cal=len(cal))
            for qlev in fdr_levels:
                sel1 = benjamini_hochberg(p1, qlev); ns1 = sel1.sum()
                fdr_F1[qlev]["selected"].append(ns1 / len(ev)); fdr_F1[qlev]["fdp"].append(float((~good_F1[ev][sel1]).mean()) if ns1 else 0.0)
                fdr_F1[qlev]["power"].append(float(sel1[good_F1[ev]].mean()))
            sel1 = benjamini_hochberg(p1, q_route)
            routing["F1_r1"].append(sel1.mean())
            routing["F1_gap_q95"].append(float(np.quantile(T["F1_gap"]["score"][ev][sel1], 0.95)) if sel1.any() else np.nan)
            routing["F1_lumo_q95"].append(float(np.quantile(T["F1_LUMO"]["score"][ev][sel1], 0.95)) if sel1.any() else np.nan)
        sel = benjamini_hochberg(p, q_route); r3 = flag[ev] & ~sel; r2 = ~sel & ~r3
        routing["r1"].append(sel.mean()); routing["r2"].append(r2.mean()); routing["r3"].append(r3.mean())
        routing["err_gap_r1_q95"].append(float(np.quantile(T["gap"]["score"][ev][sel], 0.95)) if sel.any() else np.nan)
        routing["intr_in_r1"].append(float(oracle_intr[ev][sel].mean()) if sel.any() else np.nan)
        routing["fine_in_r1"].append(float(good_fine[ev][sel].mean()) if sel.any() else np.nan)
    if verbose:
        report(df, T, res, fdr, routing, alphas, tau, tau_gross, oracle_intr, good, good_fine, fock, fdr_F1, good_F1, q_route)
    return dict(res=res, fdr=fdr, fdr_F1=fdr_F1, routing=routing, targets=T, good=good, good_fine=good_fine, good_F1=good_F1, oracle_intr=oracle_intr)

def report(df, T, res, fdr, routing, alphas, tau, tau_gross, oracle_intr, good, good_fine, fock, fdr_F1=None, good_F1=None, q_route=0.005):
    m = lambda v: float(np.nanmean(v))
    print(f"n = {len(df)} | features: {'one-Fock-build residual + spectral' if fock else 'spectral only (gap_hat, splits)'} | "
          f"oracle intruders {100*oracle_intr.mean():.1f}% | no gross failure {100*good.mean():.1f}% | all errors below fine tau {100*good_fine.mean():.1f}%")
    print("fine tau :", {k: (f"{v*1e3:.0f} mHa" if v < 1 else v) for k, v in tau.items()})
    print("gross tau:", {k: (f"{v*1e3:.0f} mHa" if v < 1 else v) for k, v in tau_gross.items()})
    for a in alphas:
        print(f"\n--- nominal coverage {100*(1-a):.0f}%  (mean over random cal/eval halves) ---")
        print(f"{'target':<12}{'marginal':>10}{'width':>9} | {'normalised':>10}{'width':>9} | {'Mondrian flag':>13}{'width(unflag)':>14} | {'Mondrian gap':>12}{'width':>9} | {'cov on intruders':>17}{'-> Mondrian':>12}")
        for k, tg in T.items():
            R_ = res[k][a]; u = tg["unit"]
            print(f"{k:<12}{100*m(R_['marg']):>9.1f}%{u*m(R_['w_marg']):>9.3g} | {100*m(R_['norm']):>9.1f}%{u*m(R_['w_norm']):>9.3g} | "
                  f"{100*m(R_['mond_flag']):>12.1f}%{u*m(R_['w_mond_flag']):>14.3g} | {100*m(R_['mond_gap']):>11.1f}%{u*m(R_['w_mond_gap']):>9.3g} | "
                  f"{100*m(R_['cov_intr_marg']):>16.1f}%{100*m(R_['cov_intr_mond']):>11.1f}%")
    print("\n--- FDR-controlled declaration 'NO GROSS FAILURE' (conformal selection, Jin & Candes 2023: BH on clipped-score p-values) ---")
    print(f"{'q':>7}{'selected':>10}{'realised FDP':>14}{'power':>8}    (levels below the gross-failure base rate {100*(1-good.mean()):.1f}% are the informative ones)")
    for q, v in fdr.items():
        print(f"{q:>7.3f}{100*m(v['selected']):>9.1f}%{100*m(v['fdp']):>13.2f}%{100*m(v['power']):>7.1f}%")
    if fdr_F1 is not None:
        print(f"\n--- same declaration for the ONE-STEP-CORRECTED surrogate F1 (no gross failure: {100*good_F1.mean():.1f}% of molecules) ---")
        print(f"{'q':>7}{'selected':>10}{'realised FDP':>14}{'power':>8}")
        for q, v in fdr_F1.items():
            print(f"{q:>7.3f}{100*m(v['selected']):>9.1f}%{100*m(v['fdp']):>13.2f}%{100*m(v['power']):>7.1f}%")
    print(f"\n--- three-way routing at q = {q_route} ---")
    print(f"route 1 (ML only, certified)      : {100*m(routing['r1']):5.1f}%   |Δgap| q95 inside = {1e3*m(routing['err_gap_r1_q95']):.2f} mHa, intruders inside = {100*m(routing['intr_in_r1']):.2f}%, all fine-tau satisfied = {100*m(routing['fine_in_r1']):.1f}%")
    print(f"route 2 (warm-start SCF)          : {100*m(routing['r2']):5.1f}%")
    print(f"route 3 (flagged -> full DFT)     : {100*m(routing['r3']):5.1f}%")
    if routing["F1_r1"]:
        print(f"\n--- F1-centric routing (every molecule has paid one Fock build) ---")
        print(f"route 1' (F1 certified, q = {q_route}) : {100*m(routing['F1_r1']):5.1f}%   inside: |Δgap(F1)| q95 = {1e3*m(routing['F1_gap_q95']):.2f} mHa, |Δε_LUMO(F1)| q95 = {1e3*m(routing['F1_lumo_q95']):.2f} mHa")
        print(f"route 3' (rest -> continue SCF)   : {100*(1-m(routing['F1_r1'])):5.1f}%")


Writing co_conformal.py


In [10]:
%%writefile co_phase3.py
# === Phase 3: cost accounting and chemistry test ==========================================
import os, sys, time, math, sqlite3
import numpy as np

QH9_XC = "b3lyp5"
H2EV = 27.211386245988

def _n_orb(Z): return int(sum(5 if z <= 2 else 14 for z in Z))
def _unpack(packed, n):
    F = np.zeros((n, n)); F[np.triu_indices(n)] = packed; return F + F.T - np.diag(np.diag(F))
def _pack32(F): return memoryview(np.ascontiguousarray(F[np.triu_indices(F.shape[0])].astype(np.float32)))
def _ortho(S, thresh=1e-7):
    s, U = np.linalg.eigh(S); keep = s > thresh; return U[:, keep] / np.sqrt(s[keep])
def _solve(F, S, X):
    Ft = X.T @ F @ X; Ft = 0.5 * (Ft + Ft.T); e, Ct = np.linalg.eigh(Ft); return e, X @ Ct
def _window(e, i, tol=0.05 / H2EV):
    lo = hi = i
    while lo > 0 and e[lo] - e[lo - 1] < tol: lo -= 1
    while hi < len(e) - 1 and e[hi + 1] - e[hi] < tol: hi += 1
    return lo, hi

def build_mol(Z, pos):
    from pyscf import gto
    mol = gto.Mole(); mol.build(verbose=0, atom=[[int(z), pos[i].tolist()] for i, z in enumerate(Z)], basis="def2svp", unit="ang")
    return mol

# ---------------------------------------------------------------- A. SCF cost
def scf_cost(Z, pos, Hhat, use_df=True, grid_level=1, conv_tol=1e-9):
    """Full SCF from (i) the default initial guess, (ii) the predicted density P_hat, (iii) the density of the
    one-step-corrected F1.  Records iterations, wall time, total energy and whether all converge to the same state.
    Also times the single Fock build that the certificate costs.  Returns (row, F1)."""
    from pyscf import dft
    mol = build_mol(Z, pos); S = mol.intor_symmetric("int1e_ovlp"); n_occ = mol.nelectron // 2; X = _ortho(S)
    eh, Ch = _solve(Hhat, S, X); P_hat = 2.0 * Ch[:, :n_occ] @ Ch[:, :n_occ].T
    def mk():
        mf = dft.RKS(mol); mf.xc = QH9_XC; mf.grids.level = grid_level; mf.conv_tol = conv_tol
        return mf.density_fit() if use_df else mf
    mf = mk(); t0 = time.time(); F1 = mf.get_fock(dm=P_hat); t_fock = time.time() - t0; F1 = 0.5 * (F1 + F1.T)
    e1, C1 = _solve(F1, S, X); P_F1 = 2.0 * C1[:, :n_occ] @ C1[:, :n_occ].T
    row = dict(n_basis=int(S.shape[0]), n_occ=n_occ, t_fock_s=t_fock)
    for tag, dm0 in (("minao", None), ("P_hat", P_hat), ("P_F1", P_F1)):
        mf = mk(); t0 = time.time()
        e = mf.kernel(dm0=dm0) if dm0 is not None else mf.kernel()
        row.update({f"cycles_{tag}": int(mf.cycles), f"t_{tag}": time.time() - t0, f"E_{tag}": float(e), f"conv_{tag}": bool(mf.converged)})
        if tag == "minao":
            e_ref_homo, e_ref_lumo = mf.mo_energy[n_occ - 1], mf.mo_energy[n_occ]
        row[f"dE_{tag}_mHa"] = 1e3 * (float(e) - row["E_minao"])
    row["same_state_P_hat"] = abs(row["dE_P_hat_mHa"]) < 1e-3; row["same_state_P_F1"] = abs(row["dE_P_F1_mHa"]) < 1e-3
    return row, F1

# ---------------------------------------------------------------- B. Fukui / site ranking
def condensed_frontier_populations(H, S, n_occ, aoslice):
    """Frontier-orbital (Mulliken-condensed) Fukui indices: f- from the HOMO block, f+ from the LUMO block,
    both degeneracy-aware (block averaged).  Returns (f_minus, f_plus) per atom, and the frontier energies."""
    X = _ortho(S); e, C = _solve(H, S, X); SC = S @ C
    out = []
    for i in (n_occ - 1, n_occ):
        lo, hi = _window(e, i); blk = slice(lo, hi + 1); k = hi - lo + 1
        q_ao = np.sum(C[:, blk] * SC[:, blk], axis=1) / k                     # Mulliken population per AO, block averaged
        out.append(np.array([q_ao[a:b].sum() for (_, _, a, b) in aoslice]))
    return out[0], out[1], e[n_occ - 1], e[n_occ]

def compare_rankings(f_ref, f_pred, heavy, tol=0.02):
    """Over heavy atoms: Spearman rho; strict top-site agreement; tolerance-aware agreement (the predicted top site is
    a true top site if its reference value lies within `tol` electrons of the reference maximum -- symmetry-equivalent
    atoms would otherwise count as disagreements); and the rank the reference top site receives in the prediction."""
    from scipy.stats import spearmanr
    a, b = f_ref[heavy], f_pred[heavy]
    if len(a) < 3: return dict(rho=np.nan, top_match=np.nan, top_match_tol=np.nan, top_rank=np.nan, n_top_equiv=np.nan)
    rho = float(spearmanr(a, b).statistic) if np.std(a) > 0 and np.std(b) > 0 else np.nan
    top = int(np.argmax(a)); order = np.argsort(-b); pred_top = int(order[0])
    return dict(rho=rho, top_match=bool(pred_top == top), top_match_tol=bool(a[pred_top] >= a.max() - tol),
                top_rank=int(np.where(order == top)[0][0]), n_top_equiv=int(np.sum(a >= a.max() - tol)))

def fukui_row(Z, pos, H_ref, H_pred, F1=None):
    mol = build_mol(Z, pos); S = mol.intor_symmetric("int1e_ovlp"); n_occ = mol.nelectron // 2; aos = mol.aoslice_by_atom()
    heavy = np.asarray(Z) > 1
    fm_r, fp_r, _, _ = condensed_frontier_populations(H_ref, S, n_occ, aos)
    fm_p, fp_p, _, _ = condensed_frontier_populations(H_pred, S, n_occ, aos)
    row = dict(n_heavy=int(heavy.sum()))
    for lab, r, p in (("fminus_hat", fm_r, fm_p), ("fplus_hat", fp_r, fp_p), ("dual_hat", fp_r - fm_r, fp_p - fm_p)):
        row.update({f"{lab}_{k}": v for k, v in compare_rankings(r, p, heavy).items()})
    if F1 is not None:
        fm_1, fp_1, _, _ = condensed_frontier_populations(F1, S, n_occ, aos)
        for lab, r, p in (("fminus_F1", fm_r, fm_1), ("fplus_F1", fp_r, fp_1), ("dual_F1", fp_r - fm_r, fp_1 - fm_1)):
            row.update({f"{lab}_{k}": v for k, v in compare_rankings(r, p, heavy).items()})
    return row

# ---------------------------------------------------------------- workers
_W = {}
def _init(subset_path, pred_path, use_df, grid_level, f1_db_path):
    os.environ["OMP_NUM_THREADS"] = "1"
    from pyscf import lib; lib.num_threads(1)
    try:
        from threadpoolctl import threadpool_limits; threadpool_limits(1)
    except Exception:
        pass
    _W["sub"] = sqlite3.connect(f"file:{subset_path}?mode=ro", uri=True); _W["pred"] = sqlite3.connect(f"file:{pred_path}?mode=ro", uri=True)
    _W["df"] = use_df; _W["grid"] = grid_level

def _load(mol_id, ckpt):
    N, Zb, posb, Hb = _W["sub"].execute("SELECT N, Z, pos, Ham FROM data WHERE id=?", (int(mol_id),)).fetchone()
    Z = np.frombuffer(Zb, np.int32 if len(Zb) == 4 * N else np.int64).astype(int); pos = np.frombuffer(posb, np.float64).reshape(N, 3); n = _n_orb(Z)
    H = _unpack(np.frombuffer(Hb, np.float64), n) if len(Hb) == 8 * n * (n + 1) // 2 else np.frombuffer(Hb, np.float64).reshape(n, n)
    Hp = _W["pred"].execute("SELECT Ham FROM pred WHERE id=? AND ckpt=?", (int(mol_id), ckpt)).fetchone()[0]
    return Z, pos, 0.5 * (H + H.T), _unpack(np.frombuffer(Hp, np.float32).astype(np.float64), n)

def _job_scf(item):
    mol_id, ckpt = item
    try:
        Z, pos, H, Hhat = _load(mol_id, ckpt)
        row, F1 = scf_cost(Z, pos, Hhat, _W["df"], _W["grid"])
        row.update(fukui_row(Z, pos, H, Hhat, F1))
        return {"id": int(mol_id), "ckpt": ckpt, "n_atoms": len(Z), **row}, (int(mol_id), ckpt, bytes(_pack32(F1)))
    except Exception as exc:
        return {"id": int(mol_id), "ckpt": ckpt, "error": f"{type(exc).__name__}: {exc}"}, None

def _job_fukui(item):
    mol_id, ckpt = item
    try:
        Z, pos, H, Hhat = _load(mol_id, ckpt)
        return {"id": int(mol_id), "ckpt": ckpt, "n_atoms": len(Z), **fukui_row(Z, pos, H, Hhat)}, None
    except Exception as exc:
        return {"id": int(mol_id), "ckpt": ckpt, "error": f"{type(exc).__name__}: {exc}"}, None

def run_pool(job, subset_path, pred_path, items, out_csv, n_workers=4, use_df=True, grid_level=1, f1_db_path=None, chunk=10, label=""):
    import csv, multiprocessing as mp
    done = set()
    if os.path.exists(out_csv) and os.path.getsize(out_csv) > 0:
        with open(out_csv) as fh: done = {(int(r["id"]), r["ckpt"]) for r in csv.DictReader(fh)}
    todo = [it for it in items if (int(it[0]), it[1]) not in done]
    print(f"{label}: {len(items)} items, {len(done)} done, {len(todo)} to run with {n_workers} workers", flush=True)
    if not todo: return
    f1db = None
    if f1_db_path:
        f1db = sqlite3.connect(f1_db_path); f1db.execute("CREATE TABLE IF NOT EXISTS f1 (id INTEGER, ckpt TEXT, Ham BLOB, PRIMARY KEY (id, ckpt))")
    fh = open(out_csv, "a", newline=""); writer = None
    if done:
        with open(out_csv) as f0: writer = csv.DictWriter(fh, fieldnames=next(csv.reader(f0)), extrasaction="ignore")
    t0 = time.time(); n_ok = n_err = 0
    with mp.get_context("fork").Pool(n_workers, initializer=_init, initargs=(subset_path, pred_path, use_df, grid_level, f1_db_path)) as pool:
        for k, (row, f1rec) in enumerate(pool.imap_unordered(job, todo, chunksize=chunk)):
            if "error" in row:
                n_err += 1; print("  [skip]", row["id"], row["ckpt"], row["error"][:120]); continue
            if writer is None:
                writer = csv.DictWriter(fh, fieldnames=list(row.keys())); writer.writeheader()
            writer.writerow(row); n_ok += 1
            if f1db is not None and f1rec is not None:
                f1db.execute("INSERT OR REPLACE INTO f1 VALUES (?,?,?)", f1rec)
            if (k + 1) % 100 == 0:
                fh.flush(); (f1db.commit() if f1db else None); rate = (k + 1) / (time.time() - t0)
                print(f"  {k+1}/{len(todo)}  ({rate:.2f} mol/s, ~{(len(todo)-k-1)/rate/60:.0f} min left)", flush=True)
    fh.close(); (f1db.commit() if f1db else None); (f1db.close() if f1db else None)
    print(f"{label} done: {n_ok} rows, {n_err} errors, {(time.time()-t0)/60:.1f} min -> {out_csv}")


Writing co_phase3.py


## Stage 0 — Phase 0 verification (8 molecules computed here)

In [11]:
if STAGES["phase0"] and not os.path.exists(f"{OUT}/phase0/phase0_davis_kahan.csv"):
    import co_phase0
    sys.argv = ["co_phase0", "--trials", str(DK_TRIALS), "--outdir", f"{OUT}/phase0"]; co_phase0.main()
else:
    print("stage 0: done (artifacts present)")


[SCF ] water ...
[SCF ] methanol ...
[SCF ] ethylene ...
[SCF ] formamide ...
[SCF ] acetonitrile ...
[SCF ] cyclopropane ...
[SCF ] pyridine ...
[SCF ] benzene ...

TEST 1/2 -- GAUGE INVARIANCE
molecule      operation                                s_B   s_naive(HOMO)
------------------------------------------------------------------------------
water         identity                            2.79e-08        1.11e-15
water         sign_flip_all                       2.79e-08        1.11e-15
water         random_sign_pattern                 2.79e-08        1.11e-15
water         random_rotation_in_subspace         2.79e-08        9.11e-01
water         permutation                         2.98e-08        1.00e+00
methanol      identity                            5.96e-08       -3.55e-15
methanol      sign_flip_all                       5.96e-08       -3.55e-15
methanol      random_sign_pattern                 5.96e-08       -3.55e-15
methanol      random_rotation_in_subspace         5

## Stage 1 — the QH9-stable database

In [12]:
import co_acquire as ACQ
DB_PATH = None
if DATA_SOURCE == "synthetic":
    DB_PATH = f"{SCRATCH}/QH9Stable_synthetic.db"
    if not ACQ.db_looks_valid(DB_PATH):
        import co_phase0 as P0
        from pyscf import gto, dft
        conn = sqlite3.connect(DB_PATH); cur = conn.cursor(); cur.execute("DROP TABLE IF EXISTS data")
        cur.execute("CREATE TABLE data (id INTEGER NOT NULL PRIMARY KEY, N INTEGER, Z BLOB, pos BLOB, Ham BLOB)")
        for i, (name, atom) in enumerate(P0.MOLECULES.items()):
            mol = gto.M(atom=atom, basis="def2svp", unit="ang", verbose=0); mf = dft.RKS(mol); mf.xc = "b3lyp5"; mf.grids.level = 3; mf.conv_tol = 1e-13; mf.kernel()
            F = mf.get_fock(); Z = np.ascontiguousarray(mol.atom_charges(), np.int32); pos = np.ascontiguousarray(mol.atom_coords(unit="ang"), np.float64)
            cur.execute("INSERT INTO data VALUES (?,?,?,?,?)", (i, len(Z), memoryview(Z), memoryview(pos), memoryview(np.ascontiguousarray(0.5*(F+F.T)))))
        conn.commit(); conn.close(); print("synthetic database written:", DB_PATH)
elif STAGES["acquire"] and resume("ConformalOrb_subset.db") is None:      # the full database is only needed until the subset exists
    attached = find_input("QH9Stable.db")
    if attached:
        DB_PATH = attached; print("[0] attached database:", DB_PATH)
    else:
        local = f"{SCRATCH}/QH9Stable.db"
        if ACQ.db_looks_valid(local):
            DB_PATH = local; print("[1] local database:", local)
        else:
            free = shutil.disk_usage(SCRATCH).free / 1e9
            mode = ACQUIRE_MODE if ACQUIRE_MODE != "auto" else ("parts" if free >= 60 else "stream" if free >= 35 else None)
            print(f"free {free:.0f} GB -> mode {mode}")
            if mode == "parts":
                DB_PATH = ACQ.get_qh9_database(SCRATCH, verify_md5=VERIFY_MD5, delete_parts=True, min_free_gb=60, n_conn=N_CONN)
            elif mode == "stream":
                urls = [(ACQ.ZENODO_URL.format(rec=ACQ.ZENODO_RECORD, name=n), md5 if VERIFY_MD5 else None) for n, (md5, gb) in ACQ.ZENODO_PARTS.items()]
                DB_PATH = ACQ.get_qh9_database_streaming(SCRATCH, urls, min_free_gb=35)
            if DB_PATH is None: raise SystemExit("database not available -- see messages above")
else:
    DB_PATH = find_input("QH9Stable.db") or (f"{SCRATCH}/QH9Stable.db" if ACQ.db_looks_valid(f"{SCRATCH}/QH9Stable.db") else None)
    print("stage 1: skipped (subset already available)" if DB_PATH is None else f"stage 1: using {DB_PATH}")


free 1100 GB -> mode parts
[3] Zenodo download; free disk in /kaggle/tmp: 1100 GB (need ~60 GB peak)
    QH9Stable.zip.001 (~8.5 GB)
    QH9Stable.zip.001: 8.52 GB in 32 segments, 0 already done, 8 connections
      1/32 segments, 1.59 GB this run, 93 MB/s
      7/32 segments, 2.97 GB this run, 87 MB/s
      11/32 segments, 3.90 GB this run, 71 MB/s
      17/32 segments, 4.96 GB this run, 69 MB/s
      20/32 segments, 5.72 GB this run, 64 MB/s
      22/32 segments, 6.24 GB this run, 59 MB/s
      23/32 segments, 6.65 GB this run, 54 MB/s
      24/32 segments, 7.30 GB this run, 44 MB/s
      26/32 segments, 7.95 GB this run, 32 MB/s
      27/32 segments, 8.22 GB this run, 29 MB/s
      28/32 segments, 8.38 GB this run, 26 MB/s
      31/32 segments, 8.51 GB this run, 25 MB/s
      done in 346 s (25 MB/s)
    md5 ok
    QH9Stable.zip.002 (~8.5 GB)
    QH9Stable.zip.002: 8.52 GB in 32 segments, 0 already done, 8 connections
      1/32 segments, 1.08 GB this run, 62 MB/s
      5/32 segments

## Stage 2 — compact subset of the official held-out splits

In [13]:
import co_subset as SUB
SUBSET_PATH = resume("ConformalOrb_subset.db")
if STAGES["subset"] and SUBSET_PATH is None:
    if DB_PATH is None: raise SystemExit("stage 2 needs the full database (stage 1)")
    SUBSET_PATH = f"{OUT}/ConformalOrb_subset.db"
    conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    rnd, ood, N_of = SUB.official_splits(conn)
    if DATA_SOURCE == "synthetic":                                          # smoke test: every molecule in both test sets
        ids = np.array([r[0] for r in conn.execute("SELECT id FROM data")]); rnd["test"], ood["test"] = ids, ids
    print("QH9-stable-id split:", {k: len(v) for k, v in rnd.items()}, "| QH9-stable-ood split:", {k: len(v) for k, v in ood.items()})
    SUB.build_subset_db(conn, SUBSET_PATH, rnd["test"], ood["test"], np.random.default_rng(SEED), n_id=SUBSET_N_ID, n_ood=SUBSET_N_OOD)
print("subset:", SUBSET_PATH, f"({os.path.getsize(SUBSET_PATH)/1e9:.2f} GB)")
sub = sqlite3.connect(f"file:{SUBSET_PATH}?mode=ro", uri=True); print(dict(sub.execute("SELECT split, COUNT(*) FROM membership GROUP BY split").fetchall()))


QH9-stable-id split: {'train': 104664, 'val': 13083, 'test': 13084} | QH9-stable-ood split: {'train': 104001, 'val': 17495, 'test': 9335}
  id_test: 1000/13084  (0.09 GB so far, 3 s)
  id_test: 2000/13084  (0.19 GB so far, 6 s)
  id_test: 3000/13084  (0.29 GB so far, 9 s)
  id_test: 4000/13084  (0.41 GB so far, 13 s)
  id_test: 5000/13084  (0.52 GB so far, 16 s)
  id_test: 6000/13084  (0.64 GB so far, 19 s)
  id_test: 7000/13084  (0.76 GB so far, 23 s)
  id_test: 8000/13084  (0.89 GB so far, 26 s)
  id_test: 9000/13084  (1.02 GB so far, 30 s)
  id_test: 10000/13084  (1.15 GB so far, 33 s)
  id_test: 11000/13084  (1.28 GB so far, 35 s)
  id_test: 12000/13084  (1.42 GB so far, 37 s)
  id_test: 13000/13084  (1.52 GB so far, 38 s)
  ood_test: 1000/9335  (1.67 GB so far, 41 s)
  ood_test: 2000/9335  (1.82 GB so far, 44 s)
  ood_test: 3000/9335  (1.96 GB so far, 47 s)
  ood_test: 4000/9335  (2.10 GB so far, 49 s)
  ood_test: 5000/9335  (2.25 GB so far, 52 s)
  ood_test: 6000/9335  (2.39 GB s

## Stage 3 — survey of the real database

In [14]:
SURVEY_CSV = resume("qh9_survey.csv")
if STAGES["survey"] and SURVEY_CSV is None:
    if DB_PATH is None:
        print("stage 3: skipped -- the full database is not present in this session (attach it or re-run stage 1)")
    else:
        import co_survey
        sys.argv = ["co_survey", "--db", DB_PATH, "--n-sample", str(N_SAMPLE if DATA_SOURCE == "qh9" else 0), "--check-scf", str(CHECK_SCF), "--out", f"{OUT}/qh9_survey.csv", "--plot"]
        co_survey.main(); SURVEY_CSV = f"{OUT}/qh9_survey.csv"
else:
    print("stage 3: done" if SURVEY_CSV else "stage 3: off")


surveying 20000 molecules from /kaggle/tmp/QH9Stable.db
  [check] id=0: max|eps_fresh - eps_stored| = 3.42e-07 Ha (converged=True)
  [check] id=1: max|eps_fresh - eps_stored| = 5.63e-07 Ha (converged=True)
  [check] id=2: max|eps_fresh - eps_stored| = 6.06e-07 Ha (converged=True)
  1000/20000  (25.4 mol/s, ~12.4 min left)
  2000/20000  (23.8 mol/s, ~12.6 min left)
  3000/20000  (22.7 mol/s, ~12.5 min left)
  4000/20000  (22.5 mol/s, ~11.8 min left)
  5000/20000  (22.3 mol/s, ~11.2 min left)
  6000/20000  (21.9 mol/s, ~10.7 min left)
  7000/20000  (21.6 mol/s, ~10.0 min left)
  8000/20000  (21.4 mol/s, ~9.4 min left)
  9000/20000  (21.1 mol/s, ~8.7 min left)
  10000/20000  (20.9 mol/s, ~8.0 min left)
  11000/20000  (20.8 mol/s, ~7.2 min left)
  12000/20000  (20.7 mol/s, ~6.4 min left)
  13000/20000  (20.6 mol/s, ~5.7 min left)
  14000/20000  (20.5 mol/s, ~4.9 min left)
  15000/20000  (20.3 mol/s, ~4.1 min left)
  16000/20000  (20.2 mol/s, ~3.3 min left)
  17000/20000  (20.1 mol/s, ~2.5 

## Stage 4 — QHNet inference (official model code and checkpoints)

In [15]:
import co_infer as INF
PRED_DB  = resume("ConformalOrb_pred.db", writable=True)
ERR_CSV  = resume("phase1b_errors.csv", writable=True)
need = {}
for ck, split in (("id", "id_test"), ("ood", "ood_test")):
    ids = [r[0] for r in sub.execute("SELECT id FROM membership WHERE split=? ORDER BY id", (split,))]
    cap = INFER_MAX.get(ck, 0); need[ck] = ids[:cap] if cap > 0 else ids
done = set()
if ERR_CSV and os.path.exists(ERR_CSV):
    with open(ERR_CSV) as fh: done = {(r["ckpt"], int(r["id"])) for r in csv.DictReader(fh)}
todo = [(ck, i) for ck in need for i in need[ck] if (ck, i) not in done]
print(f"stage 4: {sum(len(v) for v in need.values())} planned, {len(done)} done, {len(todo)} to run")
if STAGES["infer"] and todo:
    RAW = "https://raw.githubusercontent.com/divelab/AIRS/main/OpenDFT/QHBench/QH9/models/"
    for f in ("ori_QHNet_with_bias.py", "utils.py"):
        if not os.path.exists(f"models/{f}"): urllib.request.urlretrieve(RAW + f, f"models/{f}")
    open("models/__init__.py", "w").write("from .ori_QHNet_with_bias import QHNet\n")
    INF.install_shims(); torch.set_default_dtype(torch.float32); torch.manual_seed(0)
    CKPT = {"id": "QHNet-QH9-stable-id.pt", "ood": "QHNet-QH9-stable-ood.pt"}
    GD = {"id": "115MOaWWr7JNP-SJ3IMaP3Bj5pho2xlqE", "ood": "1gM02lbZCnzoAcKhbTedqjgwBSPLZXvuP"}
    ckpt_path = {}
    for ck, name in CKPT.items():
        p = find_input(name) or f"{OUT}/checkpoints/{name}"
        if not os.path.exists(p):
            import gdown; gdown.download(f"https://drive.google.com/uc?id={GD[ck]}", p, quiet=False)
        ckpt_path[ck] = p
    PRED_DB = PRED_DB or f"{OUT}/ConformalOrb_pred.db"; ERR_CSV = ERR_CSV or f"{OUT}/phase1b_errors.csv"
    pdb = sqlite3.connect(PRED_DB); pcur = pdb.cursor()
    pcur.execute("CREATE TABLE IF NOT EXISTS pred (id INTEGER, ckpt TEXT, Ham BLOB, PRIMARY KEY (id, ckpt))"); pdb.commit()
    fh = open(ERR_CSV, "a", newline=""); writer = None
    if done:
        with open(ERR_CSV) as f0: writer = csv.DictWriter(fh, fieldnames=next(csv.reader(f0)), extrasaction="ignore")
    for ck in ("id", "ood"):
        jobs = [i for (c, i) in todo if c == ck]
        if not jobs: continue
        model = INF.get_model(ckpt_path[ck], DEVICE); t0 = time.time(); checked = 0
        for k, mol_id in enumerate(jobs):
            try:
                Z, pos, H = INF.read_subset(sub, mol_id)
                Hhat, t_fwd = INF.predict_H(model, Z, pos, DEVICE, use_fast=True, check_against_official=(checked < SELF_CHECK_N)); checked += 1
                S, n_occ = INF.overlap_and_nocc(Z, pos); m = INF.frontier_metrics(H, Hhat, S, n_occ)
            except Exception as exc:
                print(f"  [skip] {ck} id={mol_id}: {type(exc).__name__}: {exc}"); continue
            row = {"id": int(mol_id), "ckpt": ck, "split": f"{ck}_test", "n_atoms": len(Z), "n_heavy": int((Z > 1).sum()), "t_fwd_ms": 1e3 * t_fwd, **m}
            if writer is None: writer = csv.DictWriter(fh, fieldnames=list(row.keys())); writer.writeheader()
            writer.writerow(row)
            pcur.execute("INSERT OR REPLACE INTO pred VALUES (?,?,?)", (int(mol_id), ck, memoryview(np.ascontiguousarray(Hhat[np.triu_indices(Hhat.shape[0])].astype(np.float32)))))
            if (k + 1) % 250 == 0:
                fh.flush(); pdb.commit(); rate = (k + 1) / (time.time() - t0); print(f"  {ck}: {k+1}/{len(jobs)} ({rate:.1f} mol/s, ~{(len(jobs)-k-1)/rate/60:.0f} min left)", flush=True)
        fh.flush(); pdb.commit(); print(f"{ck}: {len(jobs)} molecules in {(time.time()-t0)/60:.1f} min"); del model
    fh.close(); pdb.commit(); pdb.close()
d1b = pd.read_csv(ERR_CSV) if ERR_CSV and os.path.exists(ERR_CSV) else None
if d1b is not None:
    for ck in sorted(d1b.ckpt.unique()):
        x = d1b[d1b.ckpt == ck]
        print(f"{ck}: n={len(x)} | H MAE {1e6*x.H_mae.mean():.1f}e-6 Ha | eps_occ MAE {1e6*x.eps_mae_occ.mean():.0f}e-6 | psi {100*x.psi_cos_occ.mean():.2f}% | "
              f"intruders {100*((x.lumo_s_B>0.7)|(x.homo_s_B>0.7)).mean():.2f}% | ||X'dHX|| median {1e3*x.E_spec.median():.0f} mHa, vv fraction {100*x.D_vv_frac.median():.1f}%")


stage 4: 22419 planned, 0 done, 22419 to run
torch_cluster: pure-torch shim
torch_scatter: torch_geometric shim


Downloading...
From (original): https://drive.google.com/uc?id=115MOaWWr7JNP-SJ3IMaP3Bj5pho2xlqE
From (redirected): https://drive.google.com/uc?id=115MOaWWr7JNP-SJ3IMaP3Bj5pho2xlqE&confirm=t&uuid=fd27e1c8-4bb9-4749-a1d8-ae9d62574ffb
To: /kaggle/working/conformalorb/checkpoints/QHNet-QH9-stable-id.pt
100%|██████████| 83.0M/83.0M [00:00<00:00, 85.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1gM02lbZCnzoAcKhbTedqjgwBSPLZXvuP
From (redirected): https://drive.google.com/uc?id=1gM02lbZCnzoAcKhbTedqjgwBSPLZXvuP&confirm=t&uuid=9b05c940-ef13-42d6-a7aa-dfed5d59b794
To: /kaggle/working/conformalorb/checkpoints/QHNet-QH9-stable-ood.pt
100%|██████████| 83.0M/83.0M [00:00<00:00, 107MB/s]


checkpoint loaded: QHNet-QH9-stable-id.pt
  id: 250/13084 (4.5 mol/s, ~47 min left)
  id: 500/13084 (4.5 mol/s, ~46 min left)
  id: 750/13084 (4.5 mol/s, ~46 min left)
  id: 1000/13084 (4.4 mol/s, ~46 min left)
  id: 1250/13084 (4.4 mol/s, ~45 min left)
  id: 1500/13084 (4.4 mol/s, ~44 min left)
  id: 1750/13084 (4.4 mol/s, ~43 min left)
  id: 2000/13084 (4.4 mol/s, ~42 min left)
  id: 2250/13084 (4.4 mol/s, ~41 min left)
  id: 2500/13084 (4.4 mol/s, ~40 min left)
  id: 2750/13084 (4.4 mol/s, ~39 min left)
  id: 3000/13084 (4.4 mol/s, ~39 min left)
  id: 3250/13084 (4.3 mol/s, ~38 min left)
  id: 3500/13084 (4.3 mol/s, ~37 min left)
  id: 3750/13084 (4.3 mol/s, ~37 min left)
  id: 4000/13084 (4.2 mol/s, ~36 min left)
  id: 4250/13084 (4.2 mol/s, ~35 min left)
  id: 4500/13084 (4.2 mol/s, ~34 min left)
  id: 4750/13084 (4.2 mol/s, ~33 min left)
  id: 5000/13084 (4.2 mol/s, ~32 min left)
  id: 5250/13084 (4.2 mol/s, ~31 min left)
  id: 5500/13084 (4.2 mol/s, ~30 min left)
  id: 5750/1308

## Stage 5 — one Fock build per prediction (test-time features)

In [16]:
import co_fock as FOCK
FEAT_CSV = resume("phase2_features.csv", sub="phase2", writable=True) or f"{OUT}/phase2/phase2_features.csv"
if STAGES["fock"] and PRED_DB:
    pred = sqlite3.connect(f"file:{PRED_DB}?mode=ro", uri=True); items = []
    for ck in ("id", "ood"):
        ids = [int(i) for (i,) in pred.execute("SELECT id FROM pred WHERE ckpt=? ORDER BY id", (ck,))]
        cap = FOCK_MAX.get(ck, 0); items += [(i, ck) for i in (ids[:cap] if cap > 0 else ids)]
    FOCK.run_part_a(SUBSET_PATH, PRED_DB, items, FEAT_CSV, n_workers=FOCK_WORKERS, grid_level=FOCK_GRID, with_ref=True, use_df=FOCK_USE_DF)
feat = pd.read_csv(FEAT_CSV) if os.path.exists(FEAT_CSV) else None
print("features:", None if feat is None else (feat.shape, dict(feat.ckpt.value_counts())))


16000 items, 0 done, 16000 to run with 4 workers (DF J/K, grid level 1)
  200/16000  (1.62 mol/s, ~163 min left)
  400/16000  (1.63 mol/s, ~159 min left)
  600/16000  (1.51 mol/s, ~170 min left)
  800/16000  (1.30 mol/s, ~195 min left)
  1000/16000  (1.20 mol/s, ~207 min left)
  1200/16000  (1.16 mol/s, ~212 min left)
  1400/16000  (1.14 mol/s, ~214 min left)
  1600/16000  (1.13 mol/s, ~212 min left)
  1800/16000  (1.12 mol/s, ~211 min left)
  2000/16000  (1.10 mol/s, ~212 min left)
  2200/16000  (1.09 mol/s, ~212 min left)
  2400/16000  (1.08 mol/s, ~209 min left)
  2600/16000  (1.08 mol/s, ~208 min left)
  2800/16000  (1.06 mol/s, ~207 min left)
  3000/16000  (1.05 mol/s, ~207 min left)
  3200/16000  (1.03 mol/s, ~207 min left)
  3400/16000  (1.02 mol/s, ~206 min left)
  3600/16000  (1.00 mol/s, ~206 min left)
  3800/16000  (0.99 mol/s, ~206 min left)
  4000/16000  (0.97 mol/s, ~206 min left)
  4200/16000  (0.97 mol/s, ~203 min left)
  4400/16000  (0.97 mol/s, ~200 min left)
  4600/1

## Stage 6 — conformal certificates, conformal selection, routing

In [17]:
from co_conformal import *
from sklearn.metrics import roc_auc_score
RESULTS = {}; S = pd.DataFrame()
if STAGES["conformal"] and d1b is not None and feat is not None:
    df = d1b.merge(feat.drop(columns=[c for c in ("n_atoms", "n_basis", "n_occ", "gap_hat", "method") if c in feat.columns]), on=["id", "ckpt"], how="inner")
    print("merged rows:", len(df), dict(df.ckpt.value_counts()))
    for ck in sorted(df.ckpt.unique()):
        x = df[df.ckpt == ck].reset_index(drop=True)
        if len(x) < 40: print(f"{ck}: only {len(x)} rows -- skipping"); continue
        for fock in (False, True):
            print("\n" + "#" * 100 + f"\n# checkpoint {ck} | {'ONE-FOCK-BUILD features' if fock else 'spectral-only features'}\n" + "#" * 100)
            RESULTS[(ck, fock)] = run_conformal(x, alphas=ALPHAS, R=R_REPEATS, seed=0, fock=fock, tau=TAU, tau_gross=TAU_GROSS, fdr_levels=FDR_LEVELS, q_route=Q_ROUTE)
        y = ((x.lumo_s_B > 0.7) | (x.homo_s_B > 0.7)).astype(int)
        if 0 < y.sum() < len(y):
            print(f"\nintruder detection AUC ({ck}): F1-vs-Hhat s_B LUMO = {roc_auc_score(y, x.f1_sB_lumo):.3f} | global residual = {roc_auc_score(y, x.f1_res_fro):.3f} | "
                  f"frontier block = {roc_auc_score(y, x.f1_res_frontier):.3f} | predicted gap (low) = {roc_auc_score(y, -x.gap_hat):.3f}")
    summary = []
    for (ck, fock), Rz in RESULTS.items():
        for k, tg in Rz["targets"].items():
            for a in ALPHAS:
                r = Rz["res"][k][a]; u = tg["unit"]
                summary.append(dict(ckpt=ck, features="fock" if fock else "spectral", target=k, alpha=a, cov_marg=np.nanmean(r["marg"]), w_marg=u*np.nanmean(r["w_marg"]),
                                    cov_norm=np.nanmean(r["norm"]), w_norm=u*np.nanmean(r["w_norm"]), cov_mond_flag=np.nanmean(r["mond_flag"]), w_mond_flag=u*np.nanmean(r["w_mond_flag"]),
                                    cov_mond_gap=np.nanmean(r["mond_gap"]), w_mond_gap=u*np.nanmean(r["w_mond_gap"]), cov_intr_marg=np.nanmean(r["cov_intr_marg"]), cov_intr_mond=np.nanmean(r["cov_intr_mond"])))
        for q, v in Rz["fdr"].items():
            summary.append(dict(ckpt=ck, features="fock" if fock else "spectral", target="FDR", alpha=q, selected=np.nanmean(v["selected"]), fdp=np.nanmean(v["fdp"]), power=np.nanmean(v["power"])))
        if Rz.get("fdr_F1"):
            for q, v in Rz["fdr_F1"].items():
                summary.append(dict(ckpt=ck, features="fock" if fock else "spectral", target="FDR_F1", alpha=q, selected=np.nanmean(v["selected"]), fdp=np.nanmean(v["fdp"]), power=np.nanmean(v["power"])))
    S = pd.DataFrame(summary); S.to_csv(f"{OUT}/phase2/phase2_summary.csv", index=False)
else:
    print("stage 6: skipped (off or inputs missing)")


merged rows: 16000 {'id': np.int64(10000), 'ood': np.int64(6000)}

####################################################################################################
# checkpoint id | spectral-only features
####################################################################################################
n = 10000 | features: spectral only (gap_hat, splits) | oracle intruders 6.4% | no gross failure 93.4% | all errors below fine tau 83.9%
fine tau : {'eps_HOMO': '2 mHa', 'eps_LUMO': '5 mHa', 'gap': '5 mHa', 'sB_HOMO': '100 mHa', 'sB_LUMO': '200 mHa'}
gross tau: {'eps_HOMO': '10 mHa', 'eps_LUMO': '20 mHa', 'gap': '20 mHa', 'sB_HOMO': '700 mHa', 'sB_LUMO': '700 mHa'}

--- nominal coverage 90%  (mean over random cal/eval halves) ---
target        marginal    width | normalised    width | Mondrian flag width(unflag) | Mondrian gap    width |  cov on intruders -> Mondrian
eps_HOMO         90.1%     1.45 |       nan%      nan |         90.2%          1.32 |        90.1%     1.23 |       

In [18]:
for ck in ([] if len(S) == 0 else sorted(df.ckpt.unique())):
    x = df[df.ckpt == ck]; y = ((x.lumo_s_B > 0.7) | (x.homo_s_B > 0.7)).values
    fig, ax = plt.subplots(2, 3, figsize=(15, 8.5))
    ax[0,0].hist(np.log10(x.f1_res_fro[~y]), bins=60, alpha=.6, label="clean"); ax[0,0].hist(np.log10(x.f1_res_fro[y]), bins=40, alpha=.6, label="intruder (oracle)")
    ax[0,0].set_xlabel(r"$\log_{10}\|F_1-\hat H\|_F$ (Ha)"); ax[0,0].set_title(f"{ck}: one-Fock-build residual"); ax[0,0].legend()
    ax[0,1].loglog(x.ref_res_true[~y], x.f1_res_fro[~y], ".", ms=2, alpha=.3, label="clean"); ax[0,1].loglog(x.ref_res_true[y], x.f1_res_fro[y], ".", ms=3, alpha=.6, color="C3", label="intruder")
    lim = [x.ref_res_true.min()*0.8, x.f1_res_fro.max()*1.2]; ax[0,1].plot(lim, lim, "k--", lw=1); ax[0,1].set_xlabel(r"true $\|H-\hat H\|_F$"); ax[0,1].set_ylabel(r"$\|F_1-\hat H\|_F$"); ax[0,1].set_title("The residual estimates the unknown error"); ax[0,1].legend()
    ax[0,2].loglog(np.abs(x.f1_pt1_homo)*1e3+1e-4, np.abs(x.d_homo)*1e3+1e-4, ".", ms=2, alpha=.3); ax[0,2].plot([1e-3, 1e3], [1e-3, 1e3], "k--", lw=1)
    ax[0,2].set_xlabel(r"$|c_H^{\top}(F_1-\hat H)c_H|$ (mHa)"); ax[0,2].set_ylabel(r"$|\Delta\varepsilon_{\rm HOMO}|$ actual (mHa)"); ax[0,2].set_title("Test-time first-order estimate of the HOMO error")
    s95 = S[(S.ckpt == ck) & (~S.target.str.startswith("FDR")) & (S.alpha == 0.05)]
    for j, feats in enumerate(("spectral", "fock")):
        ss = s95[s95.features == feats].set_index("target")
        if len(ss) == 0: continue
        tg = [t for t in ("eps_HOMO", "eps_LUMO", "gap", "F1_HOMO", "F1_LUMO", "F1_gap") if t in ss.index]; w = np.arange(len(tg)); off = (j - .5) * .18
        ax[1,0].bar(w + off - .27, ss.loc[tg, "w_marg"], .18, label=f"{feats}: marginal"); ax[1,0].bar(w + off, ss.loc[tg, "w_mond_flag"], .18, label=f"{feats}: Mondrian(flag)"); ax[1,0].bar(w + off + .27, ss.loc[tg, "w_norm"].fillna(0), .18, label=f"{feats}: PT-normalised")
    ax[1,0].set_xticks(np.arange(6)); ax[1,0].set_xticklabels(["eps_HOMO", "eps_LUMO", "gap", "F1_HOMO", "F1_LUMO", "F1_gap"], rotation=30); ax[1,0].set_ylabel("95% certificate half-width (mHa)"); ax[1,0].set_yscale("log"); ax[1,0].legend(fontsize=6); ax[1,0].set_title("Certificate width by variant")
    for j, feats in enumerate(("spectral", "fock")):
        ss = s95[s95.features == feats].set_index("target")
        if len(ss) == 0: continue
        ax[1,1].plot(ss.index, 100*ss.cov_intr_marg, "o--", label=f"{feats}: marginal"); ax[1,1].plot(ss.index, 100*ss.cov_intr_mond, "s-", label=f"{feats}: Mondrian(flag)")
    ax[1,1].axhline(95, color="k", ls=":", lw=1); ax[1,1].set_ylabel("coverage on intruder molecules (%)"); ax[1,1].set_title("Hidden failure of marginal validity, and its repair"); ax[1,1].legend(fontsize=7); ax[1,1].tick_params(axis="x", rotation=30)
    for j, feats in enumerate(("spectral", "fock")):
        for tgt, st in (("FDR", "-"), ("FDR_F1", "--")):
            sf = S[(S.ckpt == ck) & (S.target == tgt) & (S.features == feats)].sort_values("alpha")
            if len(sf) == 0: continue
            ax[1,2].plot(sf.alpha, 100*sf.selected, "o" + st, label=f"{feats} {tgt}: selected %"); ax[1,2].plot(sf.alpha, 100*sf.fdp, "s:", label=f"{feats} {tgt}: FDP %")
    ax[1,2].plot([0.002, .05], [0.2, 5], "k:", lw=1, label="nominal FDR"); ax[1,2].set_xscale("log"); ax[1,2].set_xlabel("target FDR q"); ax[1,2].set_title("Conformal selection: 'no gross failure'"); ax[1,2].legend(fontsize=6)
    fig.tight_layout(); fig.savefig(f"{OUT}/phase2/phase2_{ck}.png", dpi=150); plt.show()


## Stage 7 — SCF cost and the chemistry test

In [19]:
import co_phase3 as P3
SCF_CSV, FUKUI_CSV, F1_DB = f"{OUT}/phase3/phase3_scf_cost.csv", f"{OUT}/phase3/phase3_fukui.csv", f"{OUT}/phase3/ConformalOrb_F1_sample.db"
for name in ("phase3_scf_cost.csv", "phase3_fukui.csv"): resume(name, sub="phase3", writable=True)
if STAGES["phase3"] and PRED_DB and d1b is not None:
    rng = np.random.default_rng(SEED); pred = sqlite3.connect(f"file:{PRED_DB}?mode=ro", uri=True); sample = []
    for ck in ("id", "ood"):
        ids = [int(i) for (i,) in pred.execute("SELECT id FROM pred WHERE ckpt=? ORDER BY id", (ck,))]
        if ids: sample += [(int(i), ck) for i in rng.choice(ids, size=min(N_SCF_PER_CKPT, len(ids)), replace=False)]
    P3.run_pool(P3._job_scf, SUBSET_PATH, PRED_DB, sample, SCF_CSV, n_workers=FOCK_WORKERS, use_df=FOCK_USE_DF, grid_level=FOCK_GRID, f1_db_path=F1_DB, label="SCF cost")
    items = [(int(i), c) for (i, c) in pred.execute("SELECT id, ckpt FROM pred ORDER BY ckpt, id")] if FUKUI_ALL else sample
    P3.run_pool(P3._job_fukui, SUBSET_PATH, PRED_DB, items, FUKUI_CSV, n_workers=FOCK_WORKERS, chunk=50, label="Fukui")
    scf = pd.read_csv(SCF_CSV); fk = pd.read_csv(FUKUI_CSV)
    intr = d1b.assign(intruder=((d1b.lumo_s_B > 0.7) | (d1b.homo_s_B > 0.7)))[["id", "ckpt", "intruder"]]
    s = scf.merge(intr, on=["id", "ckpt"], how="left")
    print("\n=== SCF COST (median; DF J/K, grid level %d) ===" % FOCK_GRID)
    for ck in sorted(s.ckpt.unique()):
        x = s[s.ckpt == ck]; cl = ~x.intruder.fillna(False).astype(bool)
        print(f"{ck}: n={len(x)} | one Fock build = {x.t_fock_s.median():.2f} s = {100*(x.t_fock_s/x.t_minao).median():.1f}% of a default SCF ({x.t_minao.median():.1f} s, {x.cycles_minao.median():.0f} it)")
        for tag in ("P_hat", "P_F1"):
            print(f"   from {tag:<6}: {x[f'cycles_{tag}'].median():.0f} it (clean {x.loc[cl, f'cycles_{tag}'].median():.0f}) | time ratio {(x[f't_{tag}']/x.t_minao).median():.2f} | same state {100*x[f'same_state_{tag}'].mean():.1f}% | converged {100*x[f'conv_{tag}'].mean():.1f}%")
    k = fk.merge(intr, on=["id", "ckpt"], how="left"); k = k[k.n_heavy >= 3]
    def agree(x, pre):
        return f"top-site (tol) {100*x[f'{pre}_top_match_tol'].mean():5.1f}% | strict {100*x[f'{pre}_top_match'].mean():5.1f}% | median rho {x[f'{pre}_rho'].median():.3f} | true top within top-2 {100*(x[f'{pre}_top_rank'] <= 1).mean():5.1f}%"
    print("\n=== FRONTIER-ORBITAL FUKUI / SITE RANKING (heavy atoms; molecules with >= 3 heavy atoms) ===")
    for ck in sorted(k.ckpt.unique()):
        x = k[k.ckpt == ck]; cl = ~x.intruder.fillna(False).astype(bool); print(f"{ck}: n={len(x)}")
        for pre, lab in (("fminus_hat", "f- Hhat"), ("fplus_hat", "f+ Hhat"), ("dual_hat", "dual Hhat")):
            print(f"  {lab:<10} all: {agree(x, pre)}\n  {'':<10} clean: {agree(x[cl], pre)}\n  {'':<10} intruder: {agree(x[~cl], pre) if (~cl).any() else 'n/a'}")
        xs = x[x.fplus_F1_rho.notna()] if "fplus_F1_rho" in x.columns else x.iloc[0:0]
        if len(xs):
            for pre, lab in (("fplus_F1", "f+ F1"), ("fminus_F1", "f- F1")): print(f"  {lab:<10} sample n={len(xs)}: {agree(xs, pre)}")
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].boxplot([s.cycles_minao, s.cycles_P_hat, s.cycles_P_F1], labels=["default", r"from $\hat P$", r"from $P(F_1)$"]); ax[0].set_ylabel("SCF iterations"); ax[0].set_title("SCF iterations by starting density")
    labs, vals = [], []
    for ck in sorted(k.ckpt.unique()):
        x = k[k.ckpt == ck]; cl2 = ~x.intruder.fillna(False).astype(bool)
        for pre, nm in (("fminus_hat", "f-"), ("fplus_hat", "f+")):
            labs += [f"{ck} {nm} clean", f"{ck} {nm} intr"]; vals += [100*x.loc[cl2, f"{pre}_top_match_tol"].mean(), 100*x.loc[~cl2, f"{pre}_top_match_tol"].mean() if (~cl2).any() else 0]
    ax[1].bar(range(len(vals)), vals, color=["C0", "C3"] * (len(vals)//2)); ax[1].set_xticks(range(len(vals))); ax[1].set_xticklabels(labs, rotation=45, ha="right", fontsize=7); ax[1].set_ylabel("top-site agreement (%)"); ax[1].set_title(r"Most reactive site preserved by $\hat H$")
    fig.tight_layout(); fig.savefig(f"{OUT}/phase3/phase3_overview.png", dpi=150); plt.show()
else:
    print("stage 7: skipped")


SCF cost: 200 items, 0 done, 200 to run with 4 workers
  100/200  (0.05 mol/s, ~33 min left)
  200/200  (0.05 mol/s, ~0 min left)
SCF cost done: 200 rows, 0 errors, 66.5 min -> /kaggle/working/conformalorb/phase3/phase3_scf_cost.csv
Fukui: 22419 items, 0 done, 22419 to run with 4 workers
  100/22419  (45.17 mol/s, ~8 min left)
  200/22419  (78.48 mol/s, ~5 min left)
  300/22419  (69.10 mol/s, ~5 min left)
  400/22419  (86.83 mol/s, ~4 min left)
  500/22419  (77.39 mol/s, ~5 min left)
  600/22419  (89.41 mol/s, ~4 min left)
  700/22419  (78.37 mol/s, ~5 min left)
  800/22419  (87.22 mol/s, ~4 min left)
  900/22419  (80.35 mol/s, ~4 min left)
  1000/22419  (86.14 mol/s, ~4 min left)
  1100/22419  (81.11 mol/s, ~4 min left)
  1200/22419  (85.66 mol/s, ~4 min left)
  1300/22419  (81.12 mol/s, ~4 min left)
  1400/22419  (85.41 mol/s, ~4 min left)
  1500/22419  (81.63 mol/s, ~4 min left)
  1600/22419  (85.64 mol/s, ~4 min left)
  1700/22419  (82.18 mol/s, ~4 min left)
  1800/22419  (85.79 mo

## Artifacts

In [20]:
for root, dirs, files in os.walk(OUT):
    for f in sorted(files):
        p = os.path.join(root, f); print(f"{os.path.getsize(p)/1e6:9.1f} MB  {p}")


   1522.8 MB  /kaggle/working/conformalorb/ConformalOrb_pred.db
   2898.6 MB  /kaggle/working/conformalorb/ConformalOrb_subset.db
     26.2 MB  /kaggle/working/conformalorb/phase1b_errors.csv
      4.9 MB  /kaggle/working/conformalorb/qh9_survey.csv
      0.3 MB  /kaggle/working/conformalorb/qh9_survey.png
      0.2 MB  /kaggle/working/conformalorb/phase0/phase0_davis_kahan.csv
      0.0 MB  /kaggle/working/conformalorb/phase0/phase0_gauge.csv
      0.0 MB  /kaggle/working/conformalorb/phase0/phase0_summary.csv
     14.1 MB  /kaggle/working/conformalorb/phase3/ConformalOrb_F1_sample.db
      2.2 MB  /kaggle/working/conformalorb/phase3/phase3_fukui.csv
      0.1 MB  /kaggle/working/conformalorb/phase3/phase3_overview.png
      0.1 MB  /kaggle/working/conformalorb/phase3/phase3_scf_cost.csv
      6.1 MB  /kaggle/working/conformalorb/phase2/phase2_features.csv
      0.4 MB  /kaggle/working/conformalorb/phase2/phase2_id.png
      0.4 MB  /kaggle/working/conformalorb/phase2/phase2_ood.png
 